# Seccion 1. Configuracion general del notebook ETL

Esta seccion inicializa el entorno base del notebook `11_load_database.ipynb` para el proceso ETL.

Incluye:
- imports necesarios del proyecto,
- constantes globales (rutas, CRS objetivo y rutas de reportes),
- carga de variables de entorno (`DB_URL`) desde `.env`,
- inicializacion de logger (`INFO`, `WARNING`, `ERROR`).

No realiza lectura de archivos, conexion a PostgreSQL ni ejecucion de SQL.

In [37]:
# ================================================================
# Imports y configuracion base
# ================================================================

import logging
import os
from pathlib import Path

from dotenv import load_dotenv
from psycopg2.extras import execute_values


# ================================================================
# Constantes globales de rutas y parametros del notebook
# ================================================================

# Directorio del notebook actual (notebooks/)
NOTEBOOK_DIR = Path.cwd()

# Raiz del proyecto (un nivel arriba de notebooks/)
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Rutas de entrada de datos
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"
IDEG_DIR = DATA_DIR / "IDEG"

# Archivos CSV limpios (staging)
HECHOS_CSV_PATH = CLEAN_DIR / "hechos_clean.csv"
VEHICULOS_INVOLUCRADOS_CSV_PATH = CLEAN_DIR / "vehiculos_involucrados_clean.csv"
FALLECIDOS_LESIONADOS_CSV_PATH = CLEAN_DIR / "fallecidos_lesionados_clean.csv"

# Diccionario oficial INE a utilizar en esta fase de carga
DICCIONARIO_VEHICULOS_PATH = (
    RAW_DIR
    / "ACCIDENTES DE TRÁNSITO - VEHICULOS INVOLUCRADOS"
    / "diccionario-vehiculos-involucrados.xlsx"
)

# Archivos GeoJSON IDEG
DEPARTAMENTOS_GEOJSON_PATH = IDEG_DIR / "agrip_03_Limites_departamentales.json"
MUNICIPIOS_GEOJSON_PATH = IDEG_DIR / "agrip_04_Limites_municipales_340.json"

# Otras rutas de soporte
SCHEMA_SQL_PATH = PROJECT_ROOT / "db" / "schema.sql"
REPORTS_DIR = PROJECT_ROOT / "reports" / "etl"

# Parametros geoespaciales
TARGET_CRS = "EPSG:4326"

# Crear carpeta de reportes para salidas de auditoria ETL
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# ================================================================
# Utilidades de configuracion (entorno y logging)
# ================================================================

def configure_etl_logger(log_level: int = logging.INFO) -> logging.Logger:
    """Configura y retorna el logger principal del notebook ETL."""
    logger_name = "etl_load_database"
    logger = logging.getLogger(logger_name)
    logger.setLevel(log_level)

    if not logger.handlers:
        stream_handler = logging.StreamHandler()
        stream_handler.setLevel(log_level)
        formatter = logging.Formatter(
            fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        )
        stream_handler.setFormatter(formatter)
        logger.addHandler(stream_handler)

    logger.propagate = False
    return logger


# Cargar variables de entorno desde .env en la raiz del proyecto
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(dotenv_path=ENV_PATH, override=False)

# Leer y validar DB_URL
DB_URL = os.getenv("DB_URL")
if not DB_URL:
    raise RuntimeError(
        "No se encontro la variable de entorno DB_URL. "
        "Verifica que exista en el archivo .env de la raiz del proyecto."
    )


# Inicializar logger del ETL
LOGGER = configure_etl_logger()
LOGGER.info("Seccion 1 inicializada correctamente.")
LOGGER.info("Configuracion base cargada y DB_URL validada.")


2026-07-12 10:08:53 | INFO | etl_load_database | Seccion 1 inicializada correctamente.
2026-07-12 10:08:53 | INFO | etl_load_database | Configuracion base cargada y DB_URL validada.


## Seccion 2. Inventario y validacion de fuentes de datos

Esta seccion valida la disponibilidad de todos los archivos obligatorios para el ETL (schema, CSV limpios, diccionarios INE y GeoJSON IDEG).

Para cada archivo se valida:
- existencia,
- ruta absoluta,
- extension,
- tamano en bytes y MB.

Si falta un archivo obligatorio, la ejecucion se detiene con un error claro.

In [4]:
# ================================================================
# Inventario y validacion de archivos fuente
# ================================================================

import pandas as pd

LOGGER.info("Iniciando validacion de inventario de fuentes...")


def resolve_raw_file(filename: str) -> Path | None:
    """Busca un archivo por nombre dentro de data/raw y retorna su ruta unica."""
    matches = sorted(RAW_DIR.rglob(filename))
    if not matches:
        return None
    if len(matches) > 1:
        raise RuntimeError(
            f"Se encontraron multiples rutas para '{filename}' en data/raw: {matches}"
        )
    return matches[0]


# Definicion de archivos obligatorios
expected_files = [
    {"tipo": "schema", "archivo": "schema.sql", "ruta_esperada": SCHEMA_SQL_PATH},
    {"tipo": "csv_clean", "archivo": "hechos_clean.csv", "ruta_esperada": HECHOS_CSV_PATH},
    {
        "tipo": "csv_clean",
        "archivo": "vehiculos_involucrados_clean.csv",
        "ruta_esperada": VEHICULOS_INVOLUCRADOS_CSV_PATH,
    },
    {
        "tipo": "csv_clean",
        "archivo": "fallecidos_lesionados_clean.csv",
        "ruta_esperada": FALLECIDOS_LESIONADOS_CSV_PATH,
    },
    {
        "tipo": "diccionario_ine",
        "archivo": "diccionario-hechos-de-transito.xlsx",
        "ruta_esperada": resolve_raw_file("diccionario-hechos-de-transito.xlsx"),
    },
    {
        "tipo": "diccionario_ine",
        "archivo": "diccionario-vehiculos-involucrados.xlsx",
        "ruta_esperada": resolve_raw_file("diccionario-vehiculos-involucrados.xlsx"),
    },
    {
        "tipo": "diccionario_ine",
        "archivo": "diccionario-fallecidos-y-lesionados.xlsx",
        "ruta_esperada": resolve_raw_file("diccionario-fallecidos-y-lesionados.xlsx"),
    },
    {
        "tipo": "geojson_ideg",
        "archivo": "agrip_03_Limites_departamentales.json",
        "ruta_esperada": DEPARTAMENTOS_GEOJSON_PATH,
    },
    {
        "tipo": "geojson_ideg",
        "archivo": "agrip_04_Limites_municipales_340.json",
        "ruta_esperada": MUNICIPIOS_GEOJSON_PATH,
    },
]

total_expected = len(expected_files)
LOGGER.info("Total de archivos esperados: %s", total_expected)

inventory_rows = []
missing_files = []

for item in expected_files:
    expected_path = item["ruta_esperada"]
    exists = expected_path is not None and expected_path.exists()

    size_bytes = expected_path.stat().st_size if exists else None
    size_mb = round(size_bytes / (1024 * 1024), 6) if size_bytes is not None else None

    inventory_rows.append(
        {
            "tipo": item["tipo"],
            "archivo": item["archivo"],
            "ruta": str(expected_path.resolve()) if expected_path is not None else None,
            "existe": exists,
            "extension": expected_path.suffix.lower() if expected_path is not None else None,
            "tamano_bytes": size_bytes,
            "tamano_mb": size_mb,
        }
    )

    if not exists:
        missing_files.append(item["archivo"])

df_inventory = pd.DataFrame(inventory_rows).sort_values(
    by=["tipo", "archivo"],
    ascending=[True, True],
).reset_index(drop=True)

found_count = int(df_inventory["existe"].sum())
missing_count = total_expected - found_count

LOGGER.info("Archivos encontrados: %s", found_count)
LOGGER.info("Archivos faltantes: %s", missing_count)

display(df_inventory)

if missing_files:
    missing_msg = (
        "Faltan archivos obligatorios para el ETL: " + ", ".join(sorted(missing_files))
    )
    LOGGER.error(missing_msg)
    raise FileNotFoundError(missing_msg)

LOGGER.info("Validacion de inventario finalizada correctamente.")


2026-07-12 10:08:58 | INFO | etl_load_database | Iniciando validacion de inventario de fuentes...
2026-07-12 10:08:58 | INFO | etl_load_database | Total de archivos esperados: 9
2026-07-12 10:08:58 | INFO | etl_load_database | Archivos encontrados: 9
2026-07-12 10:08:58 | INFO | etl_load_database | Archivos faltantes: 0


,tipo,archivo,ruta,existe,extension,tamano_bytes,tamano_mb
0,csv_clean,fallecidos_lesionados_clean.csv,C:\Users\eliza\OneDrive\Escritorio\crash-sever...,True,.csv,5333299,5.086230
1,csv_clean,hechos_clean.csv,C:\Users\eliza\OneDrive\Escritorio\crash-sever...,True,.csv,2953590,2.816763
2,csv_clean,vehiculos_involucrados_clean.csv,C:\Users\eliza\OneDrive\Escritorio\crash-sever...,True,.csv,5854546,5.583330
3,diccionario_ine,diccionario-fallecidos-y-lesionados.xlsx,C:\Users\eliza\OneDrive\Escritorio\crash-sever...,True,.xlsx,36324,0.034641
4,diccionario_ine,diccionario-hechos-de-transito.xlsx,C:\Users\eliza\OneDrive\Escritorio\crash-sever...,True,.xlsx,34223,0.032638
5,diccionario_ine,diccionario-vehiculos-involucrados.xlsx,C:\Users\eliza\OneDrive\Escritorio\crash-sever...,True,.xlsx,36303,0.034621
6,geojson_ideg,agrip_03_Limites_departamentales.json,C:\Users\eliza\OneDrive\Escritorio\crash-sever...,True,.json,10105341,9.637204
7,geojson_ideg,agrip_04_Limites_municipales_340.json,C:\Users\eliza\OneDrive\Escritorio\crash-sever...,True,.json,20854376,19.888283
8,schema,schema.sql,C:\Users\eliza\OneDrive\Escritorio\crash-sever...,True,.sql,23581,0.022489


2026-07-12 10:08:58 | INFO | etl_load_database | Validacion de inventario finalizada correctamente.


## Seccion 3. Conexion a Supabase y verificacion del entorno de base de datos

Esta seccion valida la conectividad con PostgreSQL/PostGIS en Supabase usando `DB_URL`,
y verifica precondiciones tecnicas para la carga ETL:
- conexion activa,
- extension PostGIS instalada,
- existencia de las 25 tablas esperadas del modelo.

In [5]:
# ================================================================
# Conexion y validacion de entorno PostgreSQL/PostGIS
# ================================================================

from sqlalchemy import create_engine, text

LOGGER.info("Iniciando validacion de conexion a Supabase...")

# Tablas esperadas en schema.sql (schema publico)
EXPECTED_TABLES = {
    "cat_grupo_hora",
    "cat_grupo_hora_5",
    "cat_dia_semana",
    "cat_departamento",
    "cat_municipio",
    "cat_tipo_evento",
    "cat_tipo_vehiculo",
    "cat_marca_vehiculo",
    "cat_grupo_modelo",
    "cat_modelo_vehiculo",
    "cat_color_vehiculo",
    "cat_sexo",
    "cat_grupo_edad_80",
    "cat_grupo_edad_60",
    "cat_edad_quinquenal",
    "cat_mayor_menor",
    "cat_estado_conductor",
    "cat_fall_les",
    "cat_internado",
    "hecho",
    "vehiculo",
    "vehiculo_involucrado",
    "fallecido_lesionado",
    "departamento_geom",
    "municipio_geom",
}

# Engine global para las siguientes secciones
ENGINE = create_engine(DB_URL, pool_pre_ping=True)

with ENGINE.connect() as conn:
    pg_version = conn.execute(text("SELECT version()")) .scalar_one()

    postgis_installed = conn.execute(
        text("SELECT EXISTS (SELECT 1 FROM pg_extension WHERE extname = 'postgis')")
    ).scalar_one()

    table_rows = conn.execute(
        text(
            """
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
              AND table_type = 'BASE TABLE'
            """
        )
    ).fetchall()

available_tables = {row[0] for row in table_rows}

missing_tables = sorted(EXPECTED_TABLES - available_tables)

# Tablas del modelo presentes
model_tables = available_tables.intersection(EXPECTED_TABLES)

# Tablas adicionales (normalmente PostGIS)
extra_tables = available_tables - EXPECTED_TABLES

# Separar tablas del sistema PostGIS
postgis_system_tables = {
    t for t in extra_tables
    if t.startswith("spatial_ref_sys")
}

other_extra_tables = extra_tables - postgis_system_tables

LOGGER.info("Conexion a Supabase verificada correctamente.")
LOGGER.info("Version PostgreSQL detectada: %s", pg_version)
LOGGER.info("PostGIS instalado: %s", postgis_installed)
LOGGER.info("Tablas del modelo esperadas: %s", len(EXPECTED_TABLES))
LOGGER.info("Tablas del modelo presentes: %s", len(model_tables))
LOGGER.info("Tablas del sistema PostGIS: %s", len(postgis_system_tables))

if not postgis_installed:
    LOGGER.error("La extension PostGIS no esta instalada en la base de datos.")
    raise RuntimeError("PostGIS no esta instalado en la base de datos de destino.")

if missing_tables:
    LOGGER.error("Tablas faltantes en schema public: %s", ", ".join(missing_tables))
    raise RuntimeError(
        "No se encontraron todas las tablas esperadas en schema public. "
        f"Faltantes: {', '.join(missing_tables)}"
    )

if other_extra_tables:
    LOGGER.warning(
        "Se encontraron tablas adicionales no contempladas en el modelo: %s",
        ", ".join(sorted(other_extra_tables))
    )

LOGGER.info("Validacion de entorno de base de datos finalizada correctamente.")


2026-07-12 10:09:04 | INFO | etl_load_database | Iniciando validacion de conexion a Supabase...
2026-07-12 10:09:07 | INFO | etl_load_database | Conexion a Supabase verificada correctamente.
2026-07-12 10:09:07 | INFO | etl_load_database | Version PostgreSQL detectada: PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit
2026-07-12 10:09:07 | INFO | etl_load_database | PostGIS instalado: True
2026-07-12 10:09:07 | INFO | etl_load_database | Tablas del modelo esperadas: 25
2026-07-12 10:09:07 | INFO | etl_load_database | Tablas del modelo presentes: 25
2026-07-12 10:09:07 | INFO | etl_load_database | Tablas del sistema PostGIS: 1
2026-07-12 10:09:07 | INFO | etl_load_database | Validacion de entorno de base de datos finalizada correctamente.


## Seccion 4. Lectura de fuentes y normalizacion inicial de estructuras

Esta seccion carga en memoria las fuentes de datos del ETL para preparar las siguientes fases:
- CSV limpios de staging,
- diccionarios oficiales del INE,
- GeoJSON oficiales del IDEG.

Ademas, aplica una normalizacion inicial de nombres de columnas y valida columnas requeridas por fuente.
Si falta una columna obligatoria, la ejecucion se detiene con error explicito.

In [55]:
# ================================================================
# Lectura de fuentes y validacion de columnas requeridas
# ================================================================

import geopandas as gpd
import unicodedata

LOGGER.info("Iniciando lectura de fuentes y normalizacion inicial...")


def normalize_column_name(column_name: str) -> str:
    """Normaliza nombres de columnas a formato tecnico consistente."""
    normalized = unicodedata.normalize("NFKD", str(column_name))
    normalized = normalized.encode("ascii", "ignore").decode("ascii")
    normalized = normalized.lower().strip()
    normalized = normalized.replace(" ", "_")
    return normalized


def normalize_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Retorna una copia del DataFrame con columnas normalizadas."""
    out = df.copy()
    out.columns = [normalize_column_name(c) for c in out.columns]
    return out


def ensure_required_columns(
    df: pd.DataFrame,
    required_columns: set[str],
    source_name: str,
) -> None:
    """Valida que un DataFrame contenga todas las columnas obligatorias."""
    available = set(df.columns)
    missing = sorted(required_columns - available)
    if missing:
        msg = (
            f"Fuente '{source_name}' no contiene columnas obligatorias: "
            + ", ".join(missing)
        )
        LOGGER.error(msg)
        raise ValueError(msg)


# ---------------------------------------------------------------
# 1) CSV limpios (staging)
# ---------------------------------------------------------------
df_hechos_stg_raw = pd.read_csv(HECHOS_CSV_PATH)
df_vi_stg_raw = pd.read_csv(VEHICULOS_INVOLUCRADOS_CSV_PATH)
df_fl_stg_raw = pd.read_csv(FALLECIDOS_LESIONADOS_CSV_PATH)

df_hechos_stg = normalize_dataframe_columns(df_hechos_stg_raw)
df_vi_stg = normalize_dataframe_columns(df_vi_stg_raw)
df_fl_stg = normalize_dataframe_columns(df_fl_stg_raw)

required_hechos_cols = {
    "num_corre",
    "ano_ocu",
    "dia_ocu",
    "hora_ocu",
    "g_hora",
    "g_hora_5",
    "mes_ocu",
    "dia_sem_ocu",
    "mupio_ocu",
    "depto_ocu",
    "zona_ocu",
    "tipo_veh",
    "marca_veh",
    "color_veh",
    "modelo_veh",
    "g_modelo_veh",
    "tipo_eve",
    "ano_carga",
}

required_vi_cols = {
    "num_corre",
    "ano_ocu",
    "dia_ocu",
    "hora_ocu",
    "g_hora",
    "g_hora_5",
    "mes_ocu",
    "dia_sem_ocu",
    "mupio_ocu",
    "depto_ocu",
    "zona_ocu",
    "sexo_per",
    "edad_per",
    "g_edad_80ymas",
    "g_edad_60ymas",
    "edad_quinquenales",
    "estado_con",
    "mayor_menor",
    "tipo_veh",
    "marca_veh",
    "color_veh",
    "modelo_veh",
    "g_modelo_veh",
    "tipo_eve",
    "ano_carga",
}

required_fl_cols = {
    "num_corre",
    "ano_ocu",
    "dia_ocu",
    "hora_ocu",
    "g_hora",
    "g_hora_5",
    "mes_ocu",
    "dia_sem_ocu",
    "mupio_ocu",
    "depto_ocu",
    "zona_ocu",
    "sexo_per",
    "edad_per",
    "g_edad_80ymas",
    "g_edad_60ymas",
    "edad_quinquenales",
    "mayor_menor",
    "tipo_veh",
    "marca_veh",
    "color_veh",
    "modelo_veh",
    "g_modelo_veh",
    "tipo_eve",
    "fall_les",
    "int_o_noint",
    "ano_carga",
}

ensure_required_columns(df_hechos_stg, required_hechos_cols, "hechos_clean.csv")
ensure_required_columns(df_vi_stg, required_vi_cols, "vehiculos_involucrados_clean.csv")
ensure_required_columns(df_fl_stg, required_fl_cols, "fallecidos_lesionados_clean.csv")

LOGGER.info("CSV staging cargados y validados correctamente.")
LOGGER.info("hechos_clean.csv: %s filas, %s columnas", *df_hechos_stg.shape)
LOGGER.info("vehiculos_involucrados_clean.csv: %s filas, %s columnas", *df_vi_stg.shape)
LOGGER.info("fallecidos_lesionados_clean.csv: %s filas, %s columnas", *df_fl_stg.shape)


# ---------------------------------------------------------------
# 2) Diccionarios oficiales INE
# ---------------------------------------------------------------
DICCIONARIO_HECHOS_PATH = resolve_raw_file("diccionario-hechos-de-transito.xlsx")
DICCIONARIO_FALLECIDOS_PATH = resolve_raw_file("diccionario-fallecidos-y-lesionados.xlsx")

df_dic_hechos_raw = pd.read_excel(DICCIONARIO_HECHOS_PATH)
df_dic_vehiculos_raw = pd.read_excel(DICCIONARIO_VEHICULOS_PATH)
df_dic_fallecidos_raw = pd.read_excel(DICCIONARIO_FALLECIDOS_PATH)

for dic_name, dic_df in {
    "diccionario-hechos-de-transito.xlsx": df_dic_hechos_raw,
    "diccionario-vehiculos-involucrados.xlsx": df_dic_vehiculos_raw,
    "diccionario-fallecidos-y-lesionados.xlsx": df_dic_fallecidos_raw,
}.items():
    if dic_df.shape[1] < 5:
        msg = (
            f"{dic_name} no contiene suficientes columnas para extraer "
            "variable/codigo/valor por indice."
        )
        LOGGER.error(msg)
        raise ValueError(msg)

df_dic_hechos_map = pd.DataFrame(
    {
        "variable": df_dic_hechos_raw.iloc[:, 1],
        "codigo": df_dic_hechos_raw.iloc[:, 3],
        "valor": df_dic_hechos_raw.iloc[:, 4],
    }
)
df_dic_vehiculos_map = pd.DataFrame(
    {
        "variable": df_dic_vehiculos_raw.iloc[:, 1],
        "codigo": df_dic_vehiculos_raw.iloc[:, 3],
        "valor": df_dic_vehiculos_raw.iloc[:, 4],
    }
)
df_dic_fallecidos_map = pd.DataFrame(
    {
        "variable": df_dic_fallecidos_raw.iloc[:, 1],
        "codigo": df_dic_fallecidos_raw.iloc[:, 3],
        "valor": df_dic_fallecidos_raw.iloc[:, 4],
    }
)

for dic_name, dic_map in {
    "diccionario-hechos": df_dic_hechos_map,
    "diccionario-vehiculos": df_dic_vehiculos_map,
    "diccionario-fallecidos": df_dic_fallecidos_map,
}.items():
    ensure_required_columns(dic_map, {"variable", "codigo", "valor"}, dic_name)

LOGGER.info("Diccionarios INE cargados y estructurados correctamente.")


# ---------------------------------------------------------------
# 3) GeoJSON IDEG
# ---------------------------------------------------------------
gdf_departamentos_raw = gpd.read_file(DEPARTAMENTOS_GEOJSON_PATH)
gdf_municipios_raw = gpd.read_file(MUNICIPIOS_GEOJSON_PATH)

gdf_departamentos = normalize_dataframe_columns(gdf_departamentos_raw)
gdf_municipios = normalize_dataframe_columns(gdf_municipios_raw)

ensure_required_columns(
    gdf_departamentos,
    {"cod_dep", "departamen", "geometry"},
    "agrip_03_Limites_departamentales.json",
)
ensure_required_columns(
    gdf_municipios,
    {"codigo_mun", "cod_dep", "municipio", "geometry"},
    "agrip_04_Limites_municipales_340.json",
)

LOGGER.info(
    "GeoJSON IDEG cargados y validados correctamente. Departamentos=%s, Municipios=%s",
    len(gdf_departamentos),
    len(gdf_municipios),
)

LOGGER.info("Seccion 4 finalizada correctamente.")


2026-07-13 23:04:57 | INFO | etl_load_database | Iniciando lectura de fuentes y normalizacion inicial...


2026-07-13 23:04:58 | INFO | etl_load_database | CSV staging cargados y validados correctamente.
2026-07-13 23:04:58 | INFO | etl_load_database | hechos_clean.csv: 52488 filas, 18 columnas
2026-07-13 23:04:58 | INFO | etl_load_database | vehiculos_involucrados_clean.csv: 80721 filas, 25 columnas
2026-07-13 23:04:58 | INFO | etl_load_database | fallecidos_lesionados_clean.csv: 71942 filas, 26 columnas
2026-07-13 23:04:58 | INFO | etl_load_database | Diccionarios INE cargados y estructurados correctamente.
2026-07-13 23:04:59 | INFO | etl_load_database | GeoJSON IDEG cargados y validados correctamente. Departamentos=22, Municipios=340
2026-07-13 23:04:59 | INFO | etl_load_database | Seccion 4 finalizada correctamente.


## Seccion 5. Validacion de catalogos existentes en Supabase

Esta seccion valida el estado actual de los catalogos en la base de datos:
- compara conteos esperados vs conteos actuales para catalogos ya existentes,
- identifica catalogos objetivo para completar desde fuentes oficiales,
- define la accion de carga para la siguiente fase (`VALIDAR_SOLO` o `COMPLETAR`).

No inserta ni actualiza datos en esta seccion.

In [7]:
# ================================================================
# Validacion de catalogos en Supabase (solo lectura)
# ================================================================

LOGGER.info("Iniciando validacion de catalogos existentes...")

# Catalogos indicados como existentes en el contexto aprobado
EXPECTED_EXISTING_CATALOG_COUNTS = {
    "cat_departamento": 22,
    "cat_grupo_hora": 5,
    "cat_grupo_hora_5": 4,
    "cat_dia_semana": 7,
    "cat_tipo_evento": 9,
    "cat_tipo_vehiculo": 25,
    "cat_grupo_modelo": 7,
    "cat_color_vehiculo": 18,
    "cat_sexo": 3,
    "cat_grupo_edad_80": 16,
    "cat_mayor_menor": 3,
    "cat_estado_conductor": 3,
    "cat_fall_les": 3,
    "cat_internado": 3,
}

# Catalogos definidos para completar desde fuentes oficiales
CATALOGS_TO_COMPLETE = [
    "cat_municipio",
    "cat_marca_vehiculo",
    "cat_modelo_vehiculo",
    "cat_grupo_edad_60",
    "cat_edad_quinquenal",
]

existing_catalog_rows = []
completion_catalog_rows = []

with ENGINE.connect() as conn:
    for table_name, expected_count in EXPECTED_EXISTING_CATALOG_COUNTS.items():
        actual_count = conn.execute(
            text(f"SELECT COUNT(*) FROM public.{table_name}")
        ).scalar_one()

        status = "OK" if actual_count == expected_count else "WARN"
        if status == "WARN":
            LOGGER.warning(
                "Catalogo %s: conteo esperado=%s, conteo actual=%s",
                table_name,
                expected_count,
                actual_count,
            )

        existing_catalog_rows.append(
            {
                "tabla": table_name,
                "accion": "VALIDAR_SOLO",
                "conteo_esperado": expected_count,
                "conteo_actual": int(actual_count),
                "estado": status,
            }
        )

    for table_name in CATALOGS_TO_COMPLETE:
        actual_count = conn.execute(
            text(f"SELECT COUNT(*) FROM public.{table_name}")
        ).scalar_one()

        completion_catalog_rows.append(
            {
                "tabla": table_name,
                "accion": "COMPLETAR",
                "conteo_actual": int(actual_count),
            }
        )

df_catalogos_existentes_estado = pd.DataFrame(existing_catalog_rows).sort_values(
    by=["tabla"],
    ascending=True,
).reset_index(drop=True)

df_catalogos_completar_estado = pd.DataFrame(completion_catalog_rows).sort_values(
    by=["tabla"],
    ascending=True,
).reset_index(drop=True)

existing_warn_count = int((df_catalogos_existentes_estado["estado"] == "WARN").sum())

LOGGER.info("Catalogos existentes validados: %s", len(df_catalogos_existentes_estado))
LOGGER.info("Catalogos existentes con alerta de conteo: %s", existing_warn_count)
LOGGER.info("Catalogos marcados para completar: %s", len(df_catalogos_completar_estado))

display(df_catalogos_existentes_estado)
display(df_catalogos_completar_estado)

LOGGER.info("Seccion 5 finalizada correctamente.")


2026-07-12 10:09:21 | INFO | etl_load_database | Iniciando validacion de catalogos existentes...
2026-07-12 10:09:24 | INFO | etl_load_database | Catalogos existentes validados: 14
2026-07-12 10:09:24 | INFO | etl_load_database | Catalogos existentes con alerta de conteo: 0
2026-07-12 10:09:24 | INFO | etl_load_database | Catalogos marcados para completar: 5


,tabla,accion,conteo_esperado,conteo_actual,estado
0,cat_color_vehiculo,VALIDAR_SOLO,18,18,OK
1,cat_departamento,VALIDAR_SOLO,22,22,OK
2,cat_dia_semana,VALIDAR_SOLO,7,7,OK
3,cat_estado_conductor,VALIDAR_SOLO,3,3,OK
4,cat_fall_les,VALIDAR_SOLO,3,3,OK
5,cat_grupo_edad_80,VALIDAR_SOLO,16,16,OK
6,cat_grupo_hora,VALIDAR_SOLO,5,5,OK
7,cat_grupo_hora_5,VALIDAR_SOLO,4,4,OK
8,cat_grupo_modelo,VALIDAR_SOLO,7,7,OK
9,cat_internado,VALIDAR_SOLO,3,3,OK


,tabla,accion,conteo_actual
0,cat_edad_quinquenal,COMPLETAR,18
1,cat_grupo_edad_60,COMPLETAR,12
2,cat_marca_vehiculo,COMPLETAR,0
3,cat_modelo_vehiculo,COMPLETAR,0
4,cat_municipio,COMPLETAR,0


2026-07-12 10:09:24 | INFO | etl_load_database | Seccion 5 finalizada correctamente.


## Seccion 6. Preparacion de catalogos faltantes desde INE y staging relacional

Esta seccion prepara los DataFrames fuente para los catalogos que deben completarse:
- `cat_municipio`,
- `cat_marca_vehiculo`,
- `cat_modelo_vehiculo`,
- `cat_grupo_edad_60`,
- `cat_edad_quinquenal`.

La preparacion usa el diccionario oficial del INE y, para `cat_modelo_vehiculo`,
la estructura relacional observada en staging (`vehiculos_involucrados` + `fallecidos_lesionados`).

In [19]:
# ================================================================
# Preparacion de catalogos faltantes
# ================================================================

LOGGER.info("Iniciando preparacion de catalogos faltantes...")


def normalize_text(value: object) -> str | None:
    """Normaliza texto para comparaciones tecnicas."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    txt = unicodedata.normalize("NFKD", str(value))
    txt = txt.encode("ascii", "ignore").decode("ascii")
    txt = txt.lower().strip()
    txt = txt.replace("  ", " ")
    return txt


# ---------------------------------------------------------------
# 1) Base diccionario de vehiculos: forward-fill de variable
# ---------------------------------------------------------------
df_dic_vehiculos_src = df_dic_vehiculos_map.copy()
df_dic_vehiculos_src["variable"] = (
    df_dic_vehiculos_src["variable"].map(normalize_text).ffill()
)
df_dic_vehiculos_src["codigo"] = pd.to_numeric(
    df_dic_vehiculos_src["codigo"],
    errors="coerce",
)
df_dic_vehiculos_src["valor"] = df_dic_vehiculos_src["valor"].astype(str).str.strip()
df_dic_vehiculos_src = df_dic_vehiculos_src.dropna(subset=["variable", "codigo"])


# ---------------------------------------------------------------
# 2) cat_municipio
# ---------------------------------------------------------------
df_cat_municipio_src = (
    df_dic_vehiculos_src[df_dic_vehiculos_src["variable"] == "mupio_ocu"]
    .loc[:, ["codigo", "valor"]]
    .rename(columns={"codigo": "mupio_ocu", "valor": "nombre"})
    .copy()
)
df_cat_municipio_src["mupio_ocu"] = df_cat_municipio_src["mupio_ocu"].astype(int)
df_cat_municipio_src["depto_ocu"] = (df_cat_municipio_src["mupio_ocu"] // 100).astype(int)
df_cat_municipio_src = df_cat_municipio_src.drop_duplicates(subset=["mupio_ocu"]).sort_values(
    by=["mupio_ocu"]
).reset_index(drop=True)
ensure_required_columns(
    df_cat_municipio_src,
    {"mupio_ocu", "depto_ocu", "nombre"},
    "df_cat_municipio_src",
)


# ---------------------------------------------------------------
# 3) cat_marca_vehiculo
# ---------------------------------------------------------------
df_cat_marca_vehiculo_src = (
    df_dic_vehiculos_src[df_dic_vehiculos_src["variable"] == "marca_veh"]
    .loc[:, ["codigo", "valor"]]
    .rename(columns={"codigo": "marca_veh", "valor": "nombre"})
    .copy()
)
df_cat_marca_vehiculo_src["marca_veh"] = df_cat_marca_vehiculo_src["marca_veh"].astype(int)
df_cat_marca_vehiculo_src = df_cat_marca_vehiculo_src.drop_duplicates(
    subset=["marca_veh"]
).sort_values(by=["marca_veh"]).reset_index(drop=True)

marcas_dic = set(df_cat_marca_vehiculo_src["marca_veh"].astype(int))

marcas_staging = set(
    pd.concat(
        [
            df_vi_stg["marca_veh"],
            df_fl_stg["marca_veh"],
        ],
        ignore_index=True,
    )
    .dropna()
    .astype(int)
)

faltantes = sorted(marcas_staging - marcas_dic)

if faltantes:

    LOGGER.warning(
        "%s marcas observadas en staging no existen en el diccionario INE.",
        len(faltantes),
    )

    display(faltantes)

    df_extra = pd.DataFrame(
        {
            "marca_veh": faltantes,
            "nombre": ["SIN DESCRIPCION"] * len(faltantes),
        }
    )

    df_cat_marca_vehiculo_src = (
        pd.concat(
            [df_cat_marca_vehiculo_src, df_extra],
            ignore_index=True,
        )
        .drop_duplicates(subset=["marca_veh"])
        .sort_values("marca_veh")
        .reset_index(drop=True)
    )

ensure_required_columns(
    df_cat_marca_vehiculo_src,
    {"marca_veh", "nombre"},
    "df_cat_marca_vehiculo_src",
)


# ---------------------------------------------------------------
# 4) cat_grupo_edad_60 y cat_edad_quinquenal
# ---------------------------------------------------------------
df_cat_grupo_edad_60_src = (
    df_dic_vehiculos_src[df_dic_vehiculos_src["variable"] == "g_edad_60ymas"]
    .loc[:, ["codigo", "valor"]]
    .rename(columns={"codigo": "g_edad_60ymas", "valor": "descripcion"})
    .copy()
)
df_cat_grupo_edad_60_src["g_edad_60ymas"] = df_cat_grupo_edad_60_src["g_edad_60ymas"].astype(int)
df_cat_grupo_edad_60_src = df_cat_grupo_edad_60_src.drop_duplicates(
    subset=["g_edad_60ymas"]
).sort_values(by=["g_edad_60ymas"]).reset_index(drop=True)

with ENGINE.connect() as conn:
    df_cat_grupo_edad_80_ref = pd.read_sql(
        text("SELECT g_edad_80ymas, descripcion FROM public.cat_grupo_edad_80 ORDER BY g_edad_80ymas"),
        conn,
    )

df_cat_grupo_edad_80_ref["descripcion_norm"] = df_cat_grupo_edad_80_ref["descripcion"].map(normalize_text)


def map_g60_to_g80(g60_code: int, g60_desc: str) -> int | None:
    """Mapea grupo edad 60 a grupo edad 80 usando codigos y descripciones oficiales."""
    desc_norm = normalize_text(g60_desc)
    if g60_code in set(range(1, 11)):
        return g60_code
    if desc_norm is not None and "ignorado" in desc_norm:
        return int(
            df_cat_grupo_edad_80_ref.loc[
                df_cat_grupo_edad_80_ref["descripcion_norm"].str.contains("ignorado", na=False),
                "g_edad_80ymas",
            ].iloc[0]
        )
    if desc_norm is not None and "60" in desc_norm:
        return int(
            df_cat_grupo_edad_80_ref.loc[
                df_cat_grupo_edad_80_ref["descripcion_norm"].str.startswith("60", na=False),
                "g_edad_80ymas",
            ].iloc[0]
        )
    return None


df_cat_grupo_edad_60_src["g_edad_80ymas"] = df_cat_grupo_edad_60_src.apply(
    lambda row: map_g60_to_g80(int(row["g_edad_60ymas"]), str(row["descripcion"])),
    axis=1,
)

if df_cat_grupo_edad_60_src["g_edad_80ymas"].isna().any():
    msg = "No fue posible mapear todos los codigos de g_edad_60ymas a cat_grupo_edad_80." 
    LOGGER.error(msg)
    raise ValueError(msg)

df_cat_grupo_edad_60_src["g_edad_80ymas"] = df_cat_grupo_edad_60_src["g_edad_80ymas"].astype(int)
ensure_required_columns(
    df_cat_grupo_edad_60_src,
    {"g_edad_60ymas", "g_edad_80ymas", "descripcion"},
    "df_cat_grupo_edad_60_src",
)

df_cat_edad_quinquenal_src = (
    df_dic_vehiculos_src[df_dic_vehiculos_src["variable"] == "edad_quinquenales"]
    .loc[:, ["codigo", "valor"]]
    .rename(columns={"codigo": "edad_quinquenales", "valor": "descripcion"})
    .copy()
)
df_cat_edad_quinquenal_src["edad_quinquenales"] = df_cat_edad_quinquenal_src["edad_quinquenales"].astype(int)
df_cat_edad_quinquenal_src = df_cat_edad_quinquenal_src.drop_duplicates(
    subset=["edad_quinquenales"]
).sort_values(by=["edad_quinquenales"]).reset_index(drop=True)


def map_quinquenal_to_g60(desc: str) -> int | None:
    """Asigna g_edad_60ymas a partir de descripcion quinquenal oficial."""
    desc_norm = normalize_text(desc) or ""
    if "ignorado" in desc_norm:
        return 12
    if "0 - 4" in desc_norm or "5 - 9" in desc_norm or "10 - 14" in desc_norm:
        return 1
    if "15 - 19" in desc_norm:
        return 2
    if "20 - 24" in desc_norm:
        return 3
    if "25 - 29" in desc_norm:
        return 4
    if "30 - 34" in desc_norm:
        return 5
    if "35 - 39" in desc_norm:
        return 6
    if "40 - 44" in desc_norm:
        return 7
    if "45 - 49" in desc_norm:
        return 8
    if "50 - 54" in desc_norm:
        return 9
    if "55 - 59" in desc_norm:
        return 10
    if any(token in desc_norm for token in ["60 -", "65 -", "70 -", "75 -", "80 y"]):
        return 11
    return None


df_cat_edad_quinquenal_src["g_edad_60ymas"] = df_cat_edad_quinquenal_src["descripcion"].map(map_quinquenal_to_g60)
if df_cat_edad_quinquenal_src["g_edad_60ymas"].isna().any():
    msg = "No fue posible mapear todos los codigos de edad_quinquenales a g_edad_60ymas." 
    LOGGER.error(msg)
    raise ValueError(msg)

df_cat_edad_quinquenal_src["g_edad_60ymas"] = df_cat_edad_quinquenal_src["g_edad_60ymas"].astype(int)
ensure_required_columns(
    df_cat_edad_quinquenal_src,
    {"edad_quinquenales", "g_edad_60ymas", "descripcion"},
    "df_cat_edad_quinquenal_src",
)


# ---------------------------------------------------------------
# 5) cat_modelo_vehiculo (diccionario + relacion observada en staging)
# ---------------------------------------------------------------
df_model_names_dict = (
    df_dic_vehiculos_src[df_dic_vehiculos_src["variable"] == "modelo_veh"]
    .loc[:, ["codigo", "valor"]]
    .rename(columns={"codigo": "modelo_veh", "valor": "nombre"})
    .copy()
)
df_model_names_dict["modelo_veh"] = df_model_names_dict["modelo_veh"].astype(int)
df_model_names_dict = df_model_names_dict.drop_duplicates(subset=["modelo_veh"])

df_model_rel_vi = df_vi_stg[["modelo_veh", "marca_veh", "g_modelo_veh"]].copy()
df_model_rel_fl = df_fl_stg[["modelo_veh", "marca_veh", "g_modelo_veh"]].copy()
df_model_rel = pd.concat([df_model_rel_vi, df_model_rel_fl], ignore_index=True)
for col in ["modelo_veh", "marca_veh", "g_modelo_veh"]:
    df_model_rel[col] = pd.to_numeric(df_model_rel[col], errors="coerce")
df_model_rel = df_model_rel.dropna(subset=["modelo_veh", "marca_veh", "g_modelo_veh"])
df_model_rel[["modelo_veh", "marca_veh", "g_modelo_veh"]] = df_model_rel[
    ["modelo_veh", "marca_veh", "g_modelo_veh"]
].astype(int)
df_model_rel = df_model_rel.drop_duplicates(
    subset=["modelo_veh", "marca_veh", "g_modelo_veh"]
).sort_values(by=["modelo_veh", "marca_veh", "g_modelo_veh"]).reset_index(drop=True)

# Construcción del catálogo de modelos observado en staging
df_cat_modelo_vehiculo_src = df_model_rel.merge(
    df_model_names_dict,
    on="modelo_veh",
    how="left",
)

df_cat_modelo_vehiculo_src["anio"] = None

ensure_required_columns(
    df_cat_modelo_vehiculo_src,
    {"modelo_veh", "marca_veh", "g_modelo_veh", "nombre", "anio"},
    "df_cat_modelo_vehiculo_src",
)

# ---------------------------------------------------------------
# Validación de nombres oficiales
# ---------------------------------------------------------------

df_cat_modelo_vehiculo_missing_nombre = (
    df_cat_modelo_vehiculo_src[
        df_cat_modelo_vehiculo_src["nombre"].isna()
    ]
    .copy()
)

missing_model_name_count = len(df_cat_modelo_vehiculo_missing_nombre)

if missing_model_name_count > 0:

    LOGGER.warning(
        "%s modelos observados en staging no poseen descripcion oficial en el diccionario INE.",
        missing_model_name_count,
    )

    LOGGER.warning(
        "Se conservaran estos registros asignando 'SIN DESCRIPCION' como nombre temporal."
    )

    display(df_cat_modelo_vehiculo_missing_nombre.head(20))

    df_cat_modelo_vehiculo_src["nombre_oficial"] = (
        ~df_cat_modelo_vehiculo_src["nombre"].isna()
    )

    df_cat_modelo_vehiculo_src["nombre"] = (
        df_cat_modelo_vehiculo_src["nombre"]
        .fillna("SIN DESCRIPCION")
    )

else:

    df_cat_modelo_vehiculo_src["nombre_oficial"] = True

LOGGER.info(
    "Modelos con descripcion oficial: %s",
    int(df_cat_modelo_vehiculo_src["nombre_oficial"].sum()),
)

LOGGER.info(
    "Modelos sin descripcion oficial: %s",
    missing_model_name_count,
)

LOGGER.info("Catalogos fuente preparados correctamente.")

LOGGER.info(
    "cat_municipio: %s registros",
    len(df_cat_municipio_src),
)

LOGGER.info(
    "cat_marca_vehiculo: %s registros",
    len(df_cat_marca_vehiculo_src),
)

LOGGER.info(
    "cat_grupo_edad_60: %s registros",
    len(df_cat_grupo_edad_60_src),
)

LOGGER.info(
    "cat_edad_quinquenal: %s registros",
    len(df_cat_edad_quinquenal_src),
)

LOGGER.info(
    "cat_modelo_vehiculo: %s registros",
    len(df_cat_modelo_vehiculo_src),
)

LOGGER.info("Seccion 6 finalizada correctamente.")



2026-07-12 10:19:24 | INFO | etl_load_database | Iniciando preparacion de catalogos faltantes...
2026-07-12 10:19:24 | WARNING | etl_load_database | 1 marcas observadas en staging no existen en el diccionario INE.


[82]

2026-07-12 10:19:25 | WARNING | etl_load_database | 1838 modelos observados en staging no poseen descripcion oficial en el diccionario INE.
2026-07-12 10:19:25 | WARNING | etl_load_database | Se conservaran estos registros asignando 'SIN DESCRIPCION' como nombre temporal.


,modelo_veh,marca_veh,g_modelo_veh,nombre,anio
0,1897,31,1,NaN,None
1,1954,30,99,NaN,None
2,1958,34,1,NaN,None
3,1965,14,1,NaN,None
4,1970,19,1,NaN,None
5,1972,10,1,NaN,None
6,1972,17,1,NaN,None
7,1973,14,1,NaN,None
8,1973,17,1,NaN,None
9,1973,19,1,NaN,None


2026-07-12 10:19:25 | INFO | etl_load_database | Modelos con descripcion oficial: 144
2026-07-12 10:19:25 | INFO | etl_load_database | Modelos sin descripcion oficial: 1838
2026-07-12 10:19:25 | INFO | etl_load_database | Catalogos fuente preparados correctamente.
2026-07-12 10:19:25 | INFO | etl_load_database | cat_municipio: 340 registros
2026-07-12 10:19:25 | INFO | etl_load_database | cat_marca_vehiculo: 191 registros
2026-07-12 10:19:25 | INFO | etl_load_database | cat_grupo_edad_60: 12 registros
2026-07-12 10:19:25 | INFO | etl_load_database | cat_edad_quinquenal: 18 registros
2026-07-12 10:19:25 | INFO | etl_load_database | cat_modelo_vehiculo: 1982 registros
2026-07-12 10:19:25 | INFO | etl_load_database | Seccion 6 finalizada correctamente.


In [20]:
df_model_names_dict["modelo_veh"].nunique()

df_dic_vehiculos_src["variable"].value_counts()

df_dic_vehiculos_src[
    df_dic_vehiculos_src["variable"].str.contains("modelo", case=False, na=False)
]

,variable,codigo,valor
677,modelo_veh,9999.0,Ignorado
678,g_modelo_veh,1.0,1970-1979
679,g_modelo_veh,2.0,1980-1989
680,g_modelo_veh,3.0,1990-1999
681,g_modelo_veh,4.0,2000-2009
682,g_modelo_veh,5.0,2010-2019
683,g_modelo_veh,6.0,2020-2029
684,g_modelo_veh,99.0,Ignorado


## Seccion 7. Preparacion y validacion geoespacial (IDEG -> PostGIS)

Esta seccion prepara los GeoDataFrames para carga en tablas geoespaciales:
- valida CRS de origen,
- valida geometria no vacia y geometria valida,
- reproyecta a `EPSG:4326` cuando aplica,
- aplica mapeo de columnas al modelo destino,
- valida conteos esperados (22 departamentos, 340 municipios).

In [56]:
# ================================================================
# Preparacion geoespacial para tablas PostGIS
# ================================================================

LOGGER.info("Iniciando preparacion geoespacial IDEG...")


def normalize_geo_text(value: object) -> str | None:
    """Normaliza texto geoespacial para reducir problemas de encoding."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    txt = str(value).strip()
    txt = txt.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
    txt = " ".join(txt.split())
    return txt


def ensure_valid_geodataframe(
    gdf: gpd.GeoDataFrame,
    source_name: str,
) -> None:
    """Valida geometria no vacia y validez topologica."""
    if gdf.geometry.isna().any():
        msg = f"{source_name}: existen geometria(s) nula(s)."
        LOGGER.error(msg)
        raise ValueError(msg)

    if gdf.geometry.is_empty.any():
        msg = f"{source_name}: existen geometria(s) vacia(s)."
        LOGGER.error(msg)
        raise ValueError(msg)

    invalid_count = int((~gdf.geometry.is_valid).sum())
    if invalid_count > 0:
        msg = f"{source_name}: existen {invalid_count} geometria(s) invalida(s)."
        LOGGER.error(msg)
        raise ValueError(msg)


# ---------------------------------------------------------------
# 1) Validacion de CRS y reproyeccion a EPSG:4326
# ---------------------------------------------------------------
if gdf_departamentos.crs is None:
    msg = "agrip_03_Limites_departamentales.json no define CRS."
    LOGGER.error(msg)
    raise ValueError(msg)

if gdf_municipios.crs is None:
    msg = "agrip_04_Limites_municipales_340.json no define CRS."
    LOGGER.error(msg)
    raise ValueError(msg)

LOGGER.info("CRS departamentos origen: %s", gdf_departamentos.crs)
LOGGER.info("CRS municipios origen: %s", gdf_municipios.crs)

if str(gdf_departamentos.crs).upper() != TARGET_CRS:
    gdf_departamentos_load = gdf_departamentos.to_crs(TARGET_CRS)
else:
    gdf_departamentos_load = gdf_departamentos.copy()

if str(gdf_municipios.crs).upper() != TARGET_CRS:
    gdf_municipios_load = gdf_municipios.to_crs(TARGET_CRS)
else:
    gdf_municipios_load = gdf_municipios.copy()

LOGGER.info("CRS departamentos destino: %s", gdf_departamentos_load.crs)
LOGGER.info("CRS municipios destino: %s", gdf_municipios_load.crs)

# ---------------------------------------------------------------
# 2) Mapeo de columnas al modelo destino
# ---------------------------------------------------------------
gdf_departamentos_load = gdf_departamentos_load.rename(
    columns={
        "departamen": "nombre",
    }
)
gdf_departamentos_load["cod_dep"] = pd.to_numeric(
    gdf_departamentos_load["cod_dep"],
    errors="coerce",
).astype("Int64")
gdf_departamentos_load["nombre"] = gdf_departamentos_load["nombre"].map(normalize_geo_text)
gdf_departamentos_load = gdf_departamentos_load[["cod_dep", "nombre", "geometry"]].copy()
gdf_departamentos_load = gdf_departamentos_load.rename(columns={"geometry": "geom"})
gdf_departamentos_load = gpd.GeoDataFrame(gdf_departamentos_load, geometry="geom", crs=TARGET_CRS)

gdf_municipios_load = gdf_municipios_load.rename(
    columns={
        "codigo_mun": "cod_mun",
        "municipio": "nombre",
    }
)
gdf_municipios_load["cod_mun"] = pd.to_numeric(
    gdf_municipios_load["cod_mun"],
    errors="coerce",
).astype("Int64")
gdf_municipios_load["cod_dep"] = pd.to_numeric(
    gdf_municipios_load["cod_dep"],
    errors="coerce",
).astype("Int64")
gdf_municipios_load["nombre"] = gdf_municipios_load["nombre"].map(normalize_geo_text)
gdf_municipios_load = gdf_municipios_load[["cod_mun", "cod_dep", "nombre", "geometry"]].copy()
gdf_municipios_load = gdf_municipios_load.rename(columns={"geometry": "geom"})
gdf_municipios_load = gpd.GeoDataFrame(gdf_municipios_load, geometry="geom", crs=TARGET_CRS)

ensure_required_columns(
    gdf_departamentos_load,
    {"cod_dep", "nombre", "geom"},
    "gdf_departamentos_load",
)
ensure_required_columns(
    gdf_municipios_load,
    {"cod_mun", "cod_dep", "nombre", "geom"},
    "gdf_municipios_load",
)

if gdf_departamentos_load[["cod_dep", "nombre"]].isna().any().any():
    msg = "gdf_departamentos_load contiene valores nulos en columnas obligatorias no geograficas."
    LOGGER.error(msg)
    raise ValueError(msg)

if gdf_municipios_load[["cod_mun", "cod_dep", "nombre"]].isna().any().any():
    msg = "gdf_municipios_load contiene valores nulos en columnas obligatorias no geograficas."
    LOGGER.error(msg)
    raise ValueError(msg)

gdf_departamentos_load["cod_dep"] = gdf_departamentos_load["cod_dep"].astype(int)
gdf_municipios_load["cod_mun"] = gdf_municipios_load["cod_mun"].astype(int)
gdf_municipios_load["cod_dep"] = gdf_municipios_load["cod_dep"].astype(int)

missing_dep = (
    set(gdf_municipios_load["cod_dep"])
    - set(gdf_departamentos_load["cod_dep"])
)

if missing_dep:
    msg = (
        "Existen municipios asociados a departamentos inexistentes: "
        f"{sorted(missing_dep)}"
    )
    LOGGER.error(msg)
    raise ValueError(msg)

LOGGER.info(
    "Integridad referencial municipio -> departamento validada correctamente."
)

# ---------------------------------------------------------------
# 3) Validacion de geometria y conteos esperados
# ---------------------------------------------------------------
ensure_valid_geodataframe(gdf_departamentos_load, "departamento_geom")
ensure_valid_geodataframe(gdf_municipios_load, "municipio_geom")

dep_count = len(gdf_departamentos_load)
mun_count = len(gdf_municipios_load)

if dep_count != 22:
    msg = f"Conteo invalido de departamentos: esperado=22, observado={dep_count}."
    LOGGER.error(msg)
    raise ValueError(msg)

if mun_count != 340:
    msg = f"Conteo invalido de municipios: esperado=340, observado={mun_count}."
    LOGGER.error(msg)
    raise ValueError(msg)

if gdf_departamentos_load["cod_dep"].duplicated().any():
    msg = "Se detectaron cod_dep duplicados en gdf_departamentos_load."
    LOGGER.error(msg)
    raise ValueError(msg)

if gdf_municipios_load["cod_mun"].duplicated().any():
    msg = "Se detectaron cod_mun duplicados en gdf_municipios_load."
    LOGGER.error(msg)
    raise ValueError(msg)

LOGGER.info("Geometrias preparadas correctamente: departamentos=%s, municipios=%s", dep_count, mun_count)
LOGGER.info("Seccion 7 finalizada correctamente.")


2026-07-13 23:05:45 | INFO | etl_load_database | Iniciando preparacion geoespacial IDEG...
2026-07-13 23:05:45 | INFO | etl_load_database | CRS departamentos origen: EPSG:4326
2026-07-13 23:05:45 | INFO | etl_load_database | CRS municipios origen: EPSG:4326
2026-07-13 23:05:45 | INFO | etl_load_database | CRS departamentos destino: EPSG:4326
2026-07-13 23:05:45 | INFO | etl_load_database | CRS municipios destino: EPSG:4326
2026-07-13 23:05:45 | INFO | etl_load_database | Integridad referencial municipio -> departamento validada correctamente.
2026-07-13 23:05:45 | INFO | etl_load_database | Geometrias preparadas correctamente: departamentos=22, municipios=340
2026-07-13 23:05:45 | INFO | etl_load_database | Seccion 7 finalizada correctamente.


## Seccion 8. Carga transaccional de catalogos faltantes

Esta seccion ejecuta la carga de catalogos marcados como `COMPLETAR` usando una transaccion unica de fase:
- `cat_municipio`,
- `cat_marca_vehiculo`,
- `cat_grupo_edad_60`,
- `cat_edad_quinquenal`,
- `cat_modelo_vehiculo`.

La estrategia de idempotencia es `upsert` por clave primaria.
Si ocurre un error, la transaccion completa se revierte (rollback automatico).

In [22]:
# ================================================================
# Carga transaccional de catalogos faltantes (fase catalogos)
# ================================================================

LOGGER.info("Iniciando carga transaccional de catalogos faltantes...")


def ensure_fk_subset(
    child_values: set[int],
    parent_values: set[int],
    fk_name: str,
) -> None:
    """Valida que los valores FK del hijo existan en el catalogo padre."""
    missing = sorted(child_values - parent_values)
    if missing:
        msg = f"FK no resoluble en {fk_name}. Valores faltantes: {missing[:20]}"
        LOGGER.error(msg)
        raise ValueError(msg)


with ENGINE.begin() as conn:
    # -----------------------------------------------------------
    # Prevalidaciones FK contra catalogos padre existentes
    # -----------------------------------------------------------
    ref_departamento = set(
        pd.read_sql(text("SELECT depto_ocu FROM public.cat_departamento"), conn)["depto_ocu"]
        .astype(int)
        .tolist()
    )
    ref_grupo_edad_80 = set(
        pd.read_sql(text("SELECT g_edad_80ymas FROM public.cat_grupo_edad_80"), conn)["g_edad_80ymas"]
        .astype(int)
        .tolist()
    )
    ref_grupo_modelo = set(
        pd.read_sql(text("SELECT g_modelo_veh FROM public.cat_grupo_modelo"), conn)["g_modelo_veh"]
        .astype(int)
        .tolist()
    )

    ensure_fk_subset(
        set(df_cat_municipio_src["depto_ocu"].astype(int).tolist()),
        ref_departamento,
        "cat_municipio.depto_ocu -> cat_departamento.depto_ocu",
    )
    ensure_fk_subset(
        set(df_cat_grupo_edad_60_src["g_edad_80ymas"].astype(int).tolist()),
        ref_grupo_edad_80,
        "cat_grupo_edad_60.g_edad_80ymas -> cat_grupo_edad_80.g_edad_80ymas",
    )
    ensure_fk_subset(
        set(df_cat_modelo_vehiculo_src["g_modelo_veh"].astype(int).tolist()),
        ref_grupo_modelo,
        "cat_modelo_vehiculo.g_modelo_veh -> cat_grupo_modelo.g_modelo_veh",
    )

    # -----------------------------------------------------------
    # 1) cat_municipio
    # -----------------------------------------------------------
    rows_cat_municipio = df_cat_municipio_src.to_dict("records")
    conn.execute(
        text(
            """
            INSERT INTO public.cat_municipio (mupio_ocu, depto_ocu, nombre)
            VALUES (:mupio_ocu, :depto_ocu, :nombre)
            ON CONFLICT (mupio_ocu) DO UPDATE
            SET depto_ocu = EXCLUDED.depto_ocu,
                nombre = EXCLUDED.nombre
            """
        ),
        rows_cat_municipio,
    )

    # -----------------------------------------------------------
    # 2) cat_marca_vehiculo
    # -----------------------------------------------------------
    rows_cat_marca = df_cat_marca_vehiculo_src.to_dict("records")
    conn.execute(
        text(
            """
            INSERT INTO public.cat_marca_vehiculo (marca_veh, nombre)
            VALUES (:marca_veh, :nombre)
            ON CONFLICT (marca_veh) DO UPDATE
            SET nombre = EXCLUDED.nombre
            """
        ),
        rows_cat_marca,
    )

    ref_marca = set(
        pd.read_sql(text("SELECT marca_veh FROM public.cat_marca_vehiculo"), conn)["marca_veh"]
        .astype(int)
        .tolist()
    )
    ensure_fk_subset(
        set(df_cat_modelo_vehiculo_src["marca_veh"].astype(int).tolist()),
        ref_marca,
        "cat_modelo_vehiculo.marca_veh -> cat_marca_vehiculo.marca_veh",
    )

    # -----------------------------------------------------------
    # 3) cat_grupo_edad_60
    # -----------------------------------------------------------
    rows_cat_ge60 = df_cat_grupo_edad_60_src.to_dict("records")
    conn.execute(
        text(
            """
            INSERT INTO public.cat_grupo_edad_60 (g_edad_60ymas, g_edad_80ymas, descripcion)
            VALUES (:g_edad_60ymas, :g_edad_80ymas, :descripcion)
            ON CONFLICT (g_edad_60ymas) DO UPDATE
            SET g_edad_80ymas = EXCLUDED.g_edad_80ymas,
                descripcion = EXCLUDED.descripcion
            """
        ),
        rows_cat_ge60,
    )

    ref_ge60 = set(
        pd.read_sql(text("SELECT g_edad_60ymas FROM public.cat_grupo_edad_60"), conn)["g_edad_60ymas"]
        .astype(int)
        .tolist()
    )
    ensure_fk_subset(
        set(df_cat_edad_quinquenal_src["g_edad_60ymas"].astype(int).tolist()),
        ref_ge60,
        "cat_edad_quinquenal.g_edad_60ymas -> cat_grupo_edad_60.g_edad_60ymas",
    )

    # -----------------------------------------------------------
    # 4) cat_edad_quinquenal
    # -----------------------------------------------------------
    rows_cat_eq = df_cat_edad_quinquenal_src.to_dict("records")
    conn.execute(
        text(
            """
            INSERT INTO public.cat_edad_quinquenal (edad_quinquenales, g_edad_60ymas, descripcion)
            VALUES (:edad_quinquenales, :g_edad_60ymas, :descripcion)
            ON CONFLICT (edad_quinquenales) DO UPDATE
            SET g_edad_60ymas = EXCLUDED.g_edad_60ymas,
                descripcion = EXCLUDED.descripcion
            """
        ),
        rows_cat_eq,
    )

    # -----------------------------------------------------------
    # 5) cat_modelo_vehiculo (anio siempre NULL por decision aprobada)
    # -----------------------------------------------------------
    rows_cat_modelo = df_cat_modelo_vehiculo_src.copy()
    rows_cat_modelo["anio"] = None
    rows_cat_modelo = rows_cat_modelo.to_dict("records")
    conn.execute(
        text(
            """
            INSERT INTO public.cat_modelo_vehiculo (modelo_veh, marca_veh, g_modelo_veh, nombre, anio)
            VALUES (:modelo_veh, :marca_veh, :g_modelo_veh, :nombre, :anio)
            ON CONFLICT (modelo_veh) DO UPDATE
            SET marca_veh = EXCLUDED.marca_veh,
                g_modelo_veh = EXCLUDED.g_modelo_veh,
                nombre = EXCLUDED.nombre,
                anio = EXCLUDED.anio
            """
        ),
        rows_cat_modelo,
    )

    # -----------------------------------------------------------
    # Post-validaciones de conteo y no nulos clave
    # -----------------------------------------------------------
    df_post_catalog_counts = pd.read_sql(
        text(
            """
            SELECT 'cat_municipio' AS tabla, COUNT(*) AS conteo FROM public.cat_municipio
            UNION ALL
            SELECT 'cat_marca_vehiculo' AS tabla, COUNT(*) AS conteo FROM public.cat_marca_vehiculo
            UNION ALL
            SELECT 'cat_grupo_edad_60' AS tabla, COUNT(*) AS conteo FROM public.cat_grupo_edad_60
            UNION ALL
            SELECT 'cat_edad_quinquenal' AS tabla, COUNT(*) AS conteo FROM public.cat_edad_quinquenal
            UNION ALL
            SELECT 'cat_modelo_vehiculo' AS tabla, COUNT(*) AS conteo FROM public.cat_modelo_vehiculo
            """
        ),
        conn,
    )

display(df_post_catalog_counts.sort_values(by=["tabla"]).reset_index(drop=True))

LOGGER.info("Carga transaccional de catalogos faltantes completada correctamente.")
LOGGER.info("Seccion 8 finalizada correctamente.")


2026-07-12 10:19:42 | INFO | etl_load_database | Iniciando carga transaccional de catalogos faltantes...


,tabla,conteo
0,cat_edad_quinquenal,18
1,cat_grupo_edad_60,12
2,cat_marca_vehiculo,191
3,cat_modelo_vehiculo,60
4,cat_municipio,340


2026-07-12 10:25:35 | INFO | etl_load_database | Carga transaccional de catalogos faltantes completada correctamente.
2026-07-12 10:25:35 | INFO | etl_load_database | Seccion 8 finalizada correctamente.


## Seccion 9. Carga transaccional de geometria PostGIS

Esta seccion carga las capas geoespaciales oficiales en una transaccion independiente:
- `departamento_geom`,
- `municipio_geom`.

La estrategia es idempotente mediante `upsert` por clave primaria,
con validaciones de conteo y consistencia basica posterior a la carga.

In [59]:
# ================================================================
# Carga transaccional de tablas geoespaciales
# ================================================================

from shapely.geometry import mapping

LOGGER.info("Iniciando carga transaccional de geometria PostGIS...")

with ENGINE.begin() as conn:
    # -----------------------------------------------------------
    # 1) departamento_geom
    # -----------------------------------------------------------
    dep_records = []
    for row in gdf_departamentos_load.itertuples(index=False):
        dep_records.append(
            {
                "cod_dep": int(row.cod_dep),
                "nombre": row.nombre,
                "geom_wkt": row.geom.wkt,
            }
        )

    conn.execute(
        text(
            """
            INSERT INTO public.departamento_geom (cod_dep, nombre, geom)
            VALUES (:cod_dep, :nombre, ST_SetSRID(ST_GeomFromText(:geom_wkt), 4326))
            ON CONFLICT (cod_dep) DO UPDATE
            SET nombre = EXCLUDED.nombre,
                geom = EXCLUDED.geom
            """
        ),
        dep_records,
    )

    # -----------------------------------------------------------
    # 2) municipio_geom
    # -----------------------------------------------------------
    mun_records = []
    for row in gdf_municipios_load.itertuples(index=False):
        mun_records.append(
            {
                "cod_mun": int(row.cod_mun),
                "cod_dep": int(row.cod_dep),
                "nombre": row.nombre,
                "geom_wkt": row.geom.wkt,
            }
        )

    conn.execute(
        text(
            """
            INSERT INTO public.municipio_geom (cod_mun, cod_dep, nombre, geom)
            VALUES (:cod_mun, :cod_dep, :nombre, ST_SetSRID(ST_GeomFromText(:geom_wkt), 4326))
            ON CONFLICT (cod_mun) DO UPDATE
            SET cod_dep = EXCLUDED.cod_dep,
                nombre = EXCLUDED.nombre,
                geom = EXCLUDED.geom
            """
        ),
        mun_records,
    )

    # -----------------------------------------------------------
    # Post-validaciones geoespaciales
    # -----------------------------------------------------------
    df_geom_counts = pd.read_sql(
        text(
            """
            SELECT 'departamento_geom' AS tabla, COUNT(*) AS conteo FROM public.departamento_geom
            UNION ALL
            SELECT 'municipio_geom' AS tabla, COUNT(*) AS conteo FROM public.municipio_geom
            """
        ),
        conn,
    )

    dep_count_db = int(df_geom_counts.loc[df_geom_counts["tabla"] == "departamento_geom", "conteo"].iloc[0])
    mun_count_db = int(df_geom_counts.loc[df_geom_counts["tabla"] == "municipio_geom", "conteo"].iloc[0])

    if dep_count_db != 22:
        msg = f"Conteo invalido en departamento_geom tras carga: {dep_count_db} (esperado 22)."
        LOGGER.error(msg)
        raise ValueError(msg)

    if mun_count_db != 340:
        msg = f"Conteo invalido en municipio_geom tras carga: {mun_count_db} (esperado 340)."
        LOGGER.error(msg)
        raise ValueError(msg)

    invalid_dep_geom = conn.execute(
        text("SELECT COUNT(*) FROM public.departamento_geom WHERE NOT ST_IsValid(geom)")
    ).scalar_one()
    invalid_mun_geom = conn.execute(
        text("SELECT COUNT(*) FROM public.municipio_geom WHERE NOT ST_IsValid(geom)")
    ).scalar_one()

    if int(invalid_dep_geom) > 0:
        msg = f"Se detectaron {invalid_dep_geom} geometria(s) invalida(s) en departamento_geom tras carga."
        LOGGER.error(msg)
        raise ValueError(msg)

    if int(invalid_mun_geom) > 0:
        msg = f"Se detectaron {invalid_mun_geom} geometria(s) invalida(s) en municipio_geom tras carga."
        LOGGER.error(msg)
        raise ValueError(msg)

display(df_geom_counts.sort_values(by=["tabla"]).reset_index(drop=True))

LOGGER.info("Carga geoespacial completada correctamente.")
LOGGER.info("Seccion 9 finalizada correctamente.")


2026-07-13 23:09:52 | INFO | etl_load_database | Iniciando carga transaccional de geometria PostGIS...


,tabla,conteo
0,departamento_geom,22
1,municipio_geom,340


2026-07-13 23:10:45 | INFO | etl_load_database | Carga geoespacial completada correctamente.
2026-07-13 23:10:45 | INFO | etl_load_database | Seccion 9 finalizada correctamente.


## Seccion 10. Construccion de DataFrames relacionales de carga (3FN)

Esta seccion transforma los staging CSV a estructuras alineadas al modelo relacional normalizado:
- `df_hecho_load` (solo atributos del evento),
- `df_vehiculo_load` (union VI + FL),
- `df_vi_bridge_src` y `df_fl_bridge_src` (fuentes para tablas puente).

No realiza inserciones en la base de datos en esta seccion.

In [57]:
# ================================================================
# Construccion de DataFrames de carga relacional
# ================================================================

LOGGER.info("Iniciando construccion de DataFrames relacionales de carga...")

VEHICLE_KEY_COLS = ["tipo_veh", "marca_veh", "modelo_veh", "color_veh"]


def cast_int_columns(df: pd.DataFrame, columns: list[str], source_name: str) -> pd.DataFrame:
    """Convierte columnas a int con validacion de nulos no permitidos."""
    out = df.copy()
    for col in columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")
        if out[col].isna().any():
            msg = f"{source_name}: columna {col} contiene valores no convertibles a entero."
            LOGGER.error(msg)
            raise ValueError(msg)
        out[col] = out[col].astype(int)
    return out


# ---------------------------------------------------------------
# 1) hecho: solo atributos de evento (sin atributos de vehiculo/departamento)
# ---------------------------------------------------------------
df_hecho_load = df_hechos_stg[
    [
        "num_corre",
        "ano_ocu",
        "mes_ocu",
        "dia_ocu",
        "hora_ocu",
        "zona_ocu",
        "g_hora",
        "g_hora_5",
        "dia_sem_ocu",
        "mupio_ocu",
        "tipo_eve",
    ]
].copy()
df_hecho_load = df_hecho_load.rename(columns={"ano_ocu": "anio_ocu"})
df_hecho_load = cast_int_columns(
    df_hecho_load,
    [
        "num_corre",
        "anio_ocu",
        "mes_ocu",
        "dia_ocu",
        "hora_ocu",
        "zona_ocu",
        "g_hora",
        "g_hora_5",
        "dia_sem_ocu",
        "mupio_ocu",
        "tipo_eve",
    ],
    "df_hecho_load",
)

hecho_pk_dupes = df_hecho_load.duplicated(subset=["num_corre", "anio_ocu"]).sum()
if hecho_pk_dupes > 0:
    msg = f"df_hecho_load contiene {hecho_pk_dupes} duplicados de PK (num_corre, anio_ocu)."
    LOGGER.error(msg)
    raise ValueError(msg)


# ---------------------------------------------------------------
# 2) vehiculo: union de VI + FL (decision aprobada)
# ---------------------------------------------------------------
df_vehiculo_union_src = pd.concat(
    [
        df_vi_stg[VEHICLE_KEY_COLS].copy(),
        df_fl_stg[VEHICLE_KEY_COLS].copy(),
    ],
    ignore_index=True,
)
df_vehiculo_union_src = cast_int_columns(
    df_vehiculo_union_src,
    VEHICLE_KEY_COLS,
    "df_vehiculo_union_src",
)
df_vehiculo_load = df_vehiculo_union_src.drop_duplicates(subset=VEHICLE_KEY_COLS).sort_values(
    by=VEHICLE_KEY_COLS,
    ascending=True,
).reset_index(drop=True)

vehiculo_duplicates = (
    len(df_vehiculo_union_src)
    - len(df_vehiculo_load)
)

LOGGER.info(
    "Vehiculos duplicados eliminados: %s",
    vehiculo_duplicates,
)

# ---------------------------------------------------------------
# 3) Fuentes para tablas puente (sin id_vehiculo aun)
# ---------------------------------------------------------------
df_vi_bridge_src = df_vi_stg[
    [
        "num_corre",
        "ano_ocu",
        "sexo_per",
        "edad_per",
        "edad_quinquenales",
        "mayor_menor",
        "estado_con",
        "tipo_veh",
        "marca_veh",
        "modelo_veh",
        "color_veh",
    ]
].copy()
df_vi_bridge_src = df_vi_bridge_src.rename(columns={"ano_ocu": "anio_ocu"})
df_vi_bridge_src = cast_int_columns(
    df_vi_bridge_src,
    [
        "num_corre",
        "anio_ocu",
        "sexo_per",
        "edad_per",
        "edad_quinquenales",
        "mayor_menor",
        "estado_con",
        "tipo_veh",
        "marca_veh",
        "modelo_veh",
        "color_veh",
    ],
    "df_vi_bridge_src",
)

df_fl_bridge_src = df_fl_stg[
    [
        "num_corre",
        "ano_ocu",
        "sexo_per",
        "edad_per",
        "edad_quinquenales",
        "mayor_menor",
        "fall_les",
        "int_o_noint",
        "tipo_veh",
        "marca_veh",
        "modelo_veh",
        "color_veh",
    ]
].copy()
df_fl_bridge_src = df_fl_bridge_src.rename(columns={"ano_ocu": "anio_ocu"})
df_fl_bridge_src = cast_int_columns(
    df_fl_bridge_src,
    [
        "num_corre",
        "anio_ocu",
        "sexo_per",
        "edad_per",
        "edad_quinquenales",
        "mayor_menor",
        "fall_les",
        "int_o_noint",
        "tipo_veh",
        "marca_veh",
        "modelo_veh",
        "color_veh",
    ],
    "df_fl_bridge_src",
)

# ---------------------------------------------------------------
# 4) Validacion de cobertura de vehiculo para tablas puente
# ---------------------------------------------------------------
vehiculo_key_set = set(map(tuple, df_vehiculo_load[VEHICLE_KEY_COLS].itertuples(index=False, name=None)))
vi_key_set = set(map(tuple, df_vi_bridge_src[VEHICLE_KEY_COLS].itertuples(index=False, name=None)))
fl_key_set = set(map(tuple, df_fl_bridge_src[VEHICLE_KEY_COLS].itertuples(index=False, name=None)))

missing_vi_keys = sorted(vi_key_set - vehiculo_key_set)
missing_fl_keys = sorted(fl_key_set - vehiculo_key_set)

if missing_vi_keys:
    msg = (
        "Cobertura incompleta para vehiculo en registros de vehiculo_involucrado. "
        f"Claves faltantes: {len(missing_vi_keys)}"
    )
    LOGGER.error(msg)
    raise ValueError(msg)

if missing_fl_keys:
    msg = (
        "Cobertura incompleta para vehiculo en registros de fallecido_lesionado. "
        f"Claves faltantes: {len(missing_fl_keys)}"
    )
    LOGGER.error(msg)
    raise ValueError(msg)

LOGGER.info(
    "Cobertura de vehiculos para tablas puente validada correctamente."
)

# ---------------------------------------------------------------
# Validacion de catalogos cargados
# ---------------------------------------------------------------

with ENGINE.connect() as conn:

    ref_tipo_veh = set(
        pd.read_sql(
            text("SELECT tipo_veh FROM public.cat_tipo_vehiculo"),
            conn,
        )["tipo_veh"].astype(int)
    )

    ref_marca = set(
        pd.read_sql(
            text("SELECT marca_veh FROM public.cat_marca_vehiculo"),
            conn,
        )["marca_veh"].astype(int)
    )

    ref_modelo = set(
        pd.read_sql(
            text("SELECT modelo_veh FROM public.cat_modelo_vehiculo"),
            conn,
        )["modelo_veh"].astype(int)
    )

    ref_color = set(
        pd.read_sql(
            text("SELECT color_veh FROM public.cat_color_vehiculo"),
            conn,
        )["color_veh"].astype(int)
    )
    
    ref_municipio = set(
        pd.read_sql(
            text("SELECT mupio_ocu FROM public.cat_municipio"),
            conn,
        )["mupio_ocu"].astype(int)
)

    ref_tipo_evento = set(
        pd.read_sql(
            text("SELECT tipo_eve FROM public.cat_tipo_evento"),
            conn,
        )["tipo_eve"].astype(int)
    )

    ref_grupo_hora = set(
        pd.read_sql(
            text("SELECT g_hora FROM public.cat_grupo_hora"),
            conn,
        )["g_hora"].astype(int)
    )

    ref_grupo_hora5 = set(
        pd.read_sql(
            text("SELECT g_hora_5 FROM public.cat_grupo_hora_5"),
            conn,
        )["g_hora_5"].astype(int)
    )

    ref_dia_semana = set(
        pd.read_sql(
            text("SELECT dia_sem_ocu FROM public.cat_dia_semana"),
            conn,
        )["dia_sem_ocu"].astype(int)
    )

ensure_fk_subset(
    set(df_vehiculo_load["tipo_veh"]),
    ref_tipo_veh,
    "vehiculo.tipo_veh",
)

ensure_fk_subset(
    set(df_vehiculo_load["marca_veh"]),
    ref_marca,
    "vehiculo.marca_veh",
)

ensure_fk_subset(
    set(df_vehiculo_load["modelo_veh"]),
    ref_modelo,
    "vehiculo.modelo_veh",
)

ensure_fk_subset(
    set(df_vehiculo_load["color_veh"]),
    ref_color,
    "vehiculo.color_veh",
)

ensure_fk_subset(
    set(df_hecho_load["mupio_ocu"]),
    ref_municipio,
    "hecho.mupio_ocu",
)

ensure_fk_subset(
    set(df_hecho_load["tipo_eve"]),
    ref_tipo_evento,
    "hecho.tipo_eve",
)

ensure_fk_subset(
    set(df_hecho_load["g_hora"]),
    ref_grupo_hora,
    "hecho.g_hora",
)

ensure_fk_subset(
    set(df_hecho_load["g_hora_5"]),
    ref_grupo_hora5,
    "hecho.g_hora_5",
)

ensure_fk_subset(
    set(df_hecho_load["dia_sem_ocu"]),
    ref_dia_semana,
    "hecho.dia_sem_ocu",
)

LOGGER.info(
    "Validacion contra catalogos completada correctamente."
)

LOGGER.info("df_hecho_load: %s registros", len(df_hecho_load))
LOGGER.info("df_vehiculo_load: %s registros unicos", len(df_vehiculo_load))
LOGGER.info("df_vi_bridge_src: %s registros", len(df_vi_bridge_src))
LOGGER.info("df_fl_bridge_src: %s registros", len(df_fl_bridge_src))
LOGGER.info("Seccion 10 finalizada correctamente.")


2026-07-13 23:06:08 | INFO | etl_load_database | Iniciando construccion de DataFrames relacionales de carga...
2026-07-13 23:06:08 | INFO | etl_load_database | Vehiculos duplicados eliminados: 141026
2026-07-13 23:06:08 | INFO | etl_load_database | Cobertura de vehiculos para tablas puente validada correctamente.
2026-07-13 23:06:10 | INFO | etl_load_database | Validacion contra catalogos completada correctamente.
2026-07-13 23:06:10 | INFO | etl_load_database | df_hecho_load: 52488 registros
2026-07-13 23:06:10 | INFO | etl_load_database | df_vehiculo_load: 11637 registros unicos
2026-07-13 23:06:10 | INFO | etl_load_database | df_vi_bridge_src: 80721 registros
2026-07-13 23:06:10 | INFO | etl_load_database | df_fl_bridge_src: 71942 registros
2026-07-13 23:06:10 | INFO | etl_load_database | Seccion 10 finalizada correctamente.


## Seccion 11. Carga transaccional de tablas relacionales

Esta seccion carga las tablas transaccionales del modelo en una sola transaccion de fase:
- `hecho`,
- `vehiculo`,
- `vehiculo_involucrado`,
- `fallecido_lesionado`.

Aplica recarga controlada (`DELETE` en orden de dependencias),
reinicio de secuencias y reconstruccion del mapeo `id_vehiculo` para tablas puente.

In [42]:
# ================================================================
# Carga transaccional de tablas relacionales (fase C)
# ================================================================

LOGGER.info("Iniciando carga transaccional de tablas relacionales...")


def get_sequence_name(conn, table_name: str, column_name: str) -> str:
    """Obtiene el nombre real de la secuencia asociada a una columna serial."""
    seq_name = conn.execute(
        text(
            "SELECT pg_get_serial_sequence(:full_table, :column_name)"
        ),
        {
            "full_table": f"public.{table_name}",
            "column_name": column_name,
        },
    ).scalar_one()
    if not seq_name:
        msg = f"No se encontro secuencia para public.{table_name}.{column_name}."
        LOGGER.error(msg)
        raise RuntimeError(msg)
    return seq_name


def assert_fk_values_exist(conn, source_df: pd.DataFrame, source_col: str, ref_table: str, ref_col: str) -> None:
    """Valida que todos los codigos del DataFrame existan en el catalogo de referencia."""
    source_values = set(source_df[source_col].astype(int).tolist())
    ref_values = set(
        pd.read_sql(
            text(f"SELECT {ref_col} FROM public.{ref_table}"),
            conn,
        )[ref_col]
        .astype(int)
        .tolist()
    )
    missing_values = sorted(source_values - ref_values)
    if missing_values:
        msg = (
            f"FK no resoluble: {source_col} -> {ref_table}.{ref_col}. "
            f"Valores faltantes: {missing_values[:20]}"
        )
        LOGGER.error(msg)
        raise ValueError(msg)


with ENGINE.begin() as conn:
    # -----------------------------------------------------------
    # 1) Recarga controlada por dependencias (DELETE)
    # -----------------------------------------------------------
    conn.execute(text("DELETE FROM public.fallecido_lesionado"))
    conn.execute(text("DELETE FROM public.vehiculo_involucrado"))
    conn.execute(text("DELETE FROM public.vehiculo"))
    conn.execute(text("DELETE FROM public.hecho"))

    # Reinicio de secuencias para reproducibilidad de IDs
    seq_vehiculo = get_sequence_name(conn, "vehiculo", "id_vehiculo")
    seq_vi = get_sequence_name(conn, "vehiculo_involucrado", "id_vehiculo_inv")
    seq_fl = get_sequence_name(conn, "fallecido_lesionado", "id_fall_les")

    conn.execute(text("SELECT setval(:seq_name, 1, false)"), {"seq_name": seq_vehiculo})
    conn.execute(text("SELECT setval(:seq_name, 1, false)"), {"seq_name": seq_vi})
    conn.execute(text("SELECT setval(:seq_name, 1, false)"), {"seq_name": seq_fl})

    # -----------------------------------------------------------
    # 2) Prevalidaciones FK para hecho
    # -----------------------------------------------------------
    assert_fk_values_exist(conn, df_hecho_load, "g_hora", "cat_grupo_hora", "g_hora")
    assert_fk_values_exist(conn, df_hecho_load, "g_hora_5", "cat_grupo_hora_5", "g_hora_5")
    assert_fk_values_exist(conn, df_hecho_load, "dia_sem_ocu", "cat_dia_semana", "dia_sem_ocu")
    assert_fk_values_exist(conn, df_hecho_load, "mupio_ocu", "cat_municipio", "mupio_ocu")
    assert_fk_values_exist(conn, df_hecho_load, "tipo_eve", "cat_tipo_evento", "tipo_eve")

    # -----------------------------------------------------------
    # 3) Carga de hecho
    # -----------------------------------------------------------
    
    LOGGER.info("Iniciando INSERT de hecho...")

    rows_hecho = df_hecho_load.to_dict("records")

    BATCH_SIZE = 1000

    insert_hecho = text(
        """
        INSERT INTO public.hecho (
            num_corre,
            anio_ocu,
            mes_ocu,
            dia_ocu,
            hora_ocu,
            zona_ocu,
            g_hora,
            g_hora_5,
            dia_sem_ocu,
            mupio_ocu,
            tipo_eve
        )
        VALUES (
            :num_corre,
            :anio_ocu,
            :mes_ocu,
            :dia_ocu,
            :hora_ocu,
            :zona_ocu,
            :g_hora,
            :g_hora_5,
            :dia_sem_ocu,
            :mupio_ocu,
            :tipo_eve
        )
        """
    )

    for i in range(0, len(rows_hecho), BATCH_SIZE):

        conn.execute(
            insert_hecho,
            rows_hecho[i:i+BATCH_SIZE],
        )

        LOGGER.info(
            "Hecho: %s/%s registros cargados",
            min(i + BATCH_SIZE, len(rows_hecho)),
            len(rows_hecho),
        )

    LOGGER.info("INSERT de hecho completado.")

    # -----------------------------------------------------------
    # 4) Prevalidaciones FK para vehiculo y carga
    # -----------------------------------------------------------
    assert_fk_values_exist(conn, df_vehiculo_load, "tipo_veh", "cat_tipo_vehiculo", "tipo_veh")
    assert_fk_values_exist(conn, df_vehiculo_load, "marca_veh", "cat_marca_vehiculo", "marca_veh")
    assert_fk_values_exist(conn, df_vehiculo_load, "modelo_veh", "cat_modelo_vehiculo", "modelo_veh")
    assert_fk_values_exist(conn, df_vehiculo_load, "color_veh", "cat_color_vehiculo", "color_veh")

    LOGGER.info("Iniciando INSERT de vehiculo...")

    rows_vehiculo = df_vehiculo_load.to_dict("records")

    insert_vehiculo = text(
        """
        INSERT INTO public.vehiculo (
            tipo_veh,
            marca_veh,
            modelo_veh,
            color_veh
        )
        VALUES (
            :tipo_veh,
            :marca_veh,
            :modelo_veh,
            :color_veh
        )
        """
    )

    for i in range(0, len(rows_vehiculo), BATCH_SIZE):

        conn.execute(
            insert_vehiculo,
            rows_vehiculo[i:i+BATCH_SIZE],
        )

        LOGGER.info(
            "Vehiculo: %s/%s registros cargados",
            min(i+BATCH_SIZE, len(rows_vehiculo)),
            len(rows_vehiculo),
        )

    LOGGER.info("INSERT de vehiculo completado.")

    # -----------------------------------------------------------
    # 5) Construccion de keymap id_vehiculo y armado de puentes
    # -----------------------------------------------------------
    
    
    df_vehiculo_keymap = pd.read_sql(
        text(
            """
            SELECT id_vehiculo, tipo_veh, marca_veh, modelo_veh, color_veh
            FROM public.vehiculo
            """
        ),
        conn,
    )

    df_vi_load = df_vi_bridge_src.merge(
        df_vehiculo_keymap,
        on=VEHICLE_KEY_COLS,
        how="left",
    )
    df_fl_load = df_fl_bridge_src.merge(
        df_vehiculo_keymap,
        on=VEHICLE_KEY_COLS,
        how="left",
    )

    if df_vi_load["id_vehiculo"].isna().any():
        msg = "No fue posible mapear id_vehiculo para todos los registros de vehiculo_involucrado."
        LOGGER.error(msg)
        raise ValueError(msg)

    if df_fl_load["id_vehiculo"].isna().any():
        msg = "No fue posible mapear id_vehiculo para todos los registros de fallecido_lesionado."
        LOGGER.error(msg)
        raise ValueError(msg)

    df_vi_load["id_vehiculo"] = df_vi_load["id_vehiculo"].astype(int)
    df_fl_load["id_vehiculo"] = df_fl_load["id_vehiculo"].astype(int)
    

    # -----------------------------------------------------------
    # 6)  Filtrado de registros huerfanos y validacion FK para puentes
    # -----------------------------------------------------------
    
    hechos_validos = pd.read_sql(
        text("""
            SELECT num_corre, anio_ocu
            FROM public.hecho
        """),
        conn,
    )

    antes_vi = len(df_vi_load)
    antes_fl = len(df_fl_load)

    df_vi_load = df_vi_load.merge(
        hechos_validos,
        on=["num_corre", "anio_ocu"],
        how="inner",
    )

    df_fl_load = df_fl_load.merge(
        hechos_validos,
        on=["num_corre", "anio_ocu"],
        how="inner",
    )
    
    if df_vi_load.empty:
        raise ValueError(
            "No quedaron registros para cargar en vehiculo_involucrado."
        )

    if df_fl_load.empty:
        raise ValueError(
            "No quedaron registros para cargar en fallecido_lesionado."
        )

    LOGGER.warning(
        "vehiculo_involucrado: %s registros descartados por no existir hecho asociado.",
        antes_vi - len(df_vi_load),
    )

    LOGGER.warning(
        "fallecido_lesionado: %s registros descartados por no existir hecho asociado.",
        antes_fl - len(df_fl_load),
    )
    
    LOGGER.info(
    "Integridad referencial de tablas puente preparada correctamente."
    )

    LOGGER.info(
        "vehiculo_involucrado listos para cargar: %s",
        len(df_vi_load),
    )

    LOGGER.info(
        "fallecido_lesionado listos para cargar: %s",
        len(df_fl_load),
    )

    assert_fk_values_exist(conn, df_vi_load, "sexo_per", "cat_sexo", "sexo_per")
    assert_fk_values_exist(conn, df_vi_load, "edad_quinquenales", "cat_edad_quinquenal", "edad_quinquenales")
    assert_fk_values_exist(conn, df_vi_load, "mayor_menor", "cat_mayor_menor", "mayor_menor")
    assert_fk_values_exist(conn, df_vi_load, "estado_con", "cat_estado_conductor", "estado_con")

    assert_fk_values_exist(conn, df_fl_load, "sexo_per", "cat_sexo", "sexo_per")
    assert_fk_values_exist(conn, df_fl_load, "edad_quinquenales", "cat_edad_quinquenal", "edad_quinquenales")
    assert_fk_values_exist(conn, df_fl_load, "mayor_menor", "cat_mayor_menor", "mayor_menor")
    assert_fk_values_exist(conn, df_fl_load, "fall_les", "cat_fall_les", "fall_les")
    assert_fk_values_exist(conn, df_fl_load, "int_o_noint", "cat_internado", "int_o_noint")

    # -----------------------------------------------------------
    # 7) Carga vehiculo_involucrado
    # -----------------------------------------------------------
    
    LOGGER.info("Iniciando INSERT de vehiculo_involucrado...")
    
    rows_vi = df_vi_load[
        [
            "num_corre",
            "anio_ocu",
            "id_vehiculo",
            "sexo_per",
            "edad_per",
            "edad_quinquenales",
            "mayor_menor",
            "estado_con",
        ]
    ].to_dict("records")

    insert_vi = text(
        """
        INSERT INTO public.vehiculo_involucrado (
            num_corre,
            anio_ocu,
            id_vehiculo,
            sexo_per,
            edad_per,
            edad_quinquenales,
            mayor_menor,
            estado_con
        )
        VALUES (
            :num_corre,
            :anio_ocu,
            :id_vehiculo,
            :sexo_per,
            :edad_per,
            :edad_quinquenales,
            :mayor_menor,
            :estado_con
        )
        """
    )

    for i in range(0, len(rows_vi), BATCH_SIZE):

        conn.execute(
            insert_vi,
            rows_vi[i:i+BATCH_SIZE],
        )

        LOGGER.info(
            "Vehiculo_involucrado: %s/%s",
            min(i+BATCH_SIZE, len(rows_vi)),
            len(rows_vi),
        )

    LOGGER.info("INSERT de vehiculo_involucrado completado.")

    
    # -----------------------------------------------------------
    # 8) Carga fallecido_lesionado
    # -----------------------------------------------------------
    
    LOGGER.info("Iniciando INSERT de fallecido_lesionado...")
    
    rows_fl = df_fl_load[
        [
            "num_corre",
            "anio_ocu",
            "id_vehiculo",
            "sexo_per",
            "edad_per",
            "edad_quinquenales",
            "mayor_menor",
            "fall_les",
            "int_o_noint",
        ]
    ].to_dict("records")

    insert_fl = text(
        """
        INSERT INTO public.fallecido_lesionado (
            num_corre,
            anio_ocu,
            id_vehiculo,
            sexo_per,
            edad_per,
            edad_quinquenales,
            mayor_menor,
            fall_les,
            int_o_noint
        )
        VALUES (
            :num_corre,
            :anio_ocu,
            :id_vehiculo,
            :sexo_per,
            :edad_per,
            :edad_quinquenales,
            :mayor_menor,
            :fall_les,
            :int_o_noint
        )
        """
    )

    for i in range(0, len(rows_fl), BATCH_SIZE):

        conn.execute(
            insert_fl,
            rows_fl[i:i+BATCH_SIZE],
        )

        LOGGER.info(
            "Fallecido_lesionado: %s/%s",
            min(i+BATCH_SIZE, len(rows_fl)),
            len(rows_fl),
        )


    LOGGER.info("INSERT de fallecido_lesionado completado.")
    
    # -----------------------------------------------------------
    # 9) Post-validaciones de conteo por tabla
    # -----------------------------------------------------------
    df_phase_c_counts = pd.read_sql(
        text(
            """
            SELECT 'hecho' AS tabla, COUNT(*) AS conteo FROM public.hecho
            UNION ALL
            SELECT 'vehiculo' AS tabla, COUNT(*) AS conteo FROM public.vehiculo
            UNION ALL
            SELECT 'vehiculo_involucrado' AS tabla, COUNT(*) AS conteo FROM public.vehiculo_involucrado
            UNION ALL
            SELECT 'fallecido_lesionado' AS tabla, COUNT(*) AS conteo FROM public.fallecido_lesionado
            """
        ),
        conn,
    )

display(df_phase_c_counts.sort_values(by=["tabla"]).reset_index(drop=True))

LOGGER.info("Carga transaccional de tablas relacionales completada correctamente.")
LOGGER.info("Seccion 11 finalizada correctamente.")


2026-07-12 17:44:36 | INFO | etl_load_database | Iniciando carga transaccional de tablas relacionales...
2026-07-12 17:44:40 | INFO | etl_load_database | Iniciando INSERT de hecho...
2026-07-12 17:46:56 | INFO | etl_load_database | Hecho: 1000/52488 registros cargados
2026-07-12 17:49:13 | INFO | etl_load_database | Hecho: 2000/52488 registros cargados
2026-07-12 17:51:29 | INFO | etl_load_database | Hecho: 3000/52488 registros cargados
2026-07-12 17:53:46 | INFO | etl_load_database | Hecho: 4000/52488 registros cargados
2026-07-12 17:56:03 | INFO | etl_load_database | Hecho: 5000/52488 registros cargados
2026-07-12 17:58:20 | INFO | etl_load_database | Hecho: 6000/52488 registros cargados
2026-07-12 18:00:36 | INFO | etl_load_database | Hecho: 7000/52488 registros cargados
2026-07-12 18:02:52 | INFO | etl_load_database | Hecho: 8000/52488 registros cargados
2026-07-12 18:05:09 | INFO | etl_load_database | Hecho: 9000/52488 registros cargados
2026-07-12 18:07:26 | INFO | etl_load_datab

,tabla,conteo
0,fallecido_lesionado,52483
1,hecho,52488
2,vehiculo,11637
3,vehiculo_involucrado,52488


2026-07-13 00:30:15 | INFO | etl_load_database | Carga transaccional de tablas relacionales completada correctamente.
2026-07-13 00:30:15 | INFO | etl_load_database | Seccion 11 finalizada correctamente.


## Seccion 12. Validaciones finales integrales

Esta seccion ejecuta validaciones finales de consistencia del modelo cargado:
- conteos esperados de tablas transaccionales,
- integridad referencial en tablas puente,
- consistencia geoespacial basica,
- consistencia municipio -> departamento a traves de catalogo geográfico.

In [60]:
# ================================================================
# Validaciones finales integrales del modelo cargado
# ================================================================

LOGGER.info("Iniciando validaciones finales integrales...")

final_validation_rows = []

with ENGINE.connect() as conn:
    # -----------------------------------------------------------
    # 1) Conteos esperados en tablas transaccionales
    # -----------------------------------------------------------
    count_hecho = conn.execute(text("SELECT COUNT(*) FROM public.hecho")).scalar_one()
    count_vehiculo = conn.execute(text("SELECT COUNT(*) FROM public.vehiculo")).scalar_one()
    count_vi = conn.execute(text("SELECT COUNT(*) FROM public.vehiculo_involucrado")).scalar_one()
    count_fl = conn.execute(text("SELECT COUNT(*) FROM public.fallecido_lesionado")).scalar_one()

    final_validation_rows.extend(
        [
            {
                "validation_type": "count_check",
                "object_name": "hecho",
                "result": "PASS" if int(count_hecho) == len(df_hecho_load) else "FAIL",
                "observed": int(count_hecho),
                "expected": int(len(df_hecho_load)),
            },
            {
                "validation_type": "count_check",
                "object_name": "vehiculo",
                "result": "PASS" if int(count_vehiculo) == len(df_vehiculo_load) else "FAIL",
                "observed": int(count_vehiculo),
                "expected": int(len(df_vehiculo_load)),
            },
            {
                "validation_type": "count_check",
                "object_name": "vehiculo_involucrado",
                "result": "PASS" if int(count_vi) == len(df_vi_load) else "FAIL",
                "observed": int(count_vi),
                "expected": int(len(df_vi_load)),
            },
            {
                "validation_type": "count_check",
                "object_name": "fallecido_lesionado",
                "result": "PASS" if int(count_fl) == len(df_fl_load) else "FAIL",
                "observed": int(count_fl),
                "expected": int(len(df_fl_load)),
            },
        ]
    )

    # -----------------------------------------------------------
    # 2) Integridad referencial en tablas puente
    # -----------------------------------------------------------
    vi_missing_hecho = conn.execute(
        text(
            """
            SELECT COUNT(*)
            FROM public.vehiculo_involucrado vi
            LEFT JOIN public.hecho h
              ON h.num_corre = vi.num_corre
             AND h.anio_ocu = vi.anio_ocu
            WHERE h.num_corre IS NULL
            """
        )
    ).scalar_one()
    vi_missing_vehiculo = conn.execute(
        text(
            """
            SELECT COUNT(*)
            FROM public.vehiculo_involucrado vi
            LEFT JOIN public.vehiculo v ON v.id_vehiculo = vi.id_vehiculo
            WHERE v.id_vehiculo IS NULL
            """
        )
    ).scalar_one()

    fl_missing_hecho = conn.execute(
        text(
            """
            SELECT COUNT(*)
            FROM public.fallecido_lesionado fl
            LEFT JOIN public.hecho h
              ON h.num_corre = fl.num_corre
             AND h.anio_ocu = fl.anio_ocu
            WHERE h.num_corre IS NULL
            """
        )
    ).scalar_one()
    fl_missing_vehiculo = conn.execute(
        text(
            """
            SELECT COUNT(*)
            FROM public.fallecido_lesionado fl
            LEFT JOIN public.vehiculo v ON v.id_vehiculo = fl.id_vehiculo
            WHERE v.id_vehiculo IS NULL
            """
        )
    ).scalar_one()

    final_validation_rows.extend(
        [
            {
                "validation_type": "fk_check",
                "object_name": "vehiculo_involucrado -> hecho",
                "result": "PASS" if int(vi_missing_hecho) == 0 else "FAIL",
                "observed": int(vi_missing_hecho),
                "expected": 0,
            },
            {
                "validation_type": "fk_check",
                "object_name": "vehiculo_involucrado -> vehiculo",
                "result": "PASS" if int(vi_missing_vehiculo) == 0 else "FAIL",
                "observed": int(vi_missing_vehiculo),
                "expected": 0,
            },
            {
                "validation_type": "fk_check",
                "object_name": "fallecido_lesionado -> hecho",
                "result": "PASS" if int(fl_missing_hecho) == 0 else "FAIL",
                "observed": int(fl_missing_hecho),
                "expected": 0,
            },
            {
                "validation_type": "fk_check",
                "object_name": "fallecido_lesionado -> vehiculo",
                "result": "PASS" if int(fl_missing_vehiculo) == 0 else "FAIL",
                "observed": int(fl_missing_vehiculo),
                "expected": 0,
            },
        ]
    )

    # -----------------------------------------------------------
    # 3) Integridad geoespacial basica
    # -----------------------------------------------------------
    dep_invalid_geom = conn.execute(
        text("SELECT COUNT(*) FROM public.departamento_geom WHERE NOT ST_IsValid(geom)")
    ).scalar_one()
    mun_invalid_geom = conn.execute(
        text("SELECT COUNT(*) FROM public.municipio_geom WHERE NOT ST_IsValid(geom)")
    ).scalar_one()

    final_validation_rows.extend(
        [
            {
                "validation_type": "postgis_check",
                "object_name": "departamento_geom.geometry_valid",
                "result": "PASS" if int(dep_invalid_geom) == 0 else "FAIL",
                "observed": int(dep_invalid_geom),
                "expected": 0,
            },
            {
                "validation_type": "postgis_check",
                "object_name": "municipio_geom.geometry_valid",
                "result": "PASS" if int(mun_invalid_geom) == 0 else "FAIL",
                "observed": int(mun_invalid_geom),
                "expected": 0,
            },
        ]
    )

    # -----------------------------------------------------------
    # 4) Coherencia municipio -> departamento en hechos
    # -----------------------------------------------------------
    mismatched_mupio_dep = conn.execute(
        text(
            """
            SELECT COUNT(*)
            FROM public.hecho h
            JOIN public.cat_municipio m ON m.mupio_ocu = h.mupio_ocu
            WHERE m.depto_ocu <> FLOOR(h.mupio_ocu / 100.0)::int
            """
        )
    ).scalar_one()

    final_validation_rows.append(
        {
            "validation_type": "consistency_check",
            "object_name": "hecho.mupio_ocu -> cat_municipio.depto_ocu",
            "result": "PASS" if int(mismatched_mupio_dep) == 0 else "WARN",
            "observed": int(mismatched_mupio_dep),
            "expected": 0,
        }
    )

df_final_validations = pd.DataFrame(final_validation_rows)
display(df_final_validations)

failed_count = int((df_final_validations["result"] == "FAIL").sum())
warn_count = int((df_final_validations["result"] == "WARN").sum())

LOGGER.info("Validaciones finales ejecutadas: %s", len(df_final_validations))
LOGGER.info("Resultados FAIL: %s", failed_count)
LOGGER.info("Resultados WARN: %s", warn_count)

if failed_count > 0:
    raise RuntimeError(
        "Validaciones finales con errores criticos (FAIL). Revisar df_final_validations."
    )

LOGGER.info("Seccion 12 finalizada correctamente.")


2026-07-13 23:12:01 | INFO | etl_load_database | Iniciando validaciones finales integrales...


,validation_type,object_name,result,observed,expected
0,count_check,hecho,PASS,52488,52488
1,count_check,vehiculo,PASS,11637,11637
2,count_check,vehiculo_involucrado,PASS,52488,52488
3,count_check,fallecido_lesionado,PASS,52483,52483
4,fk_check,vehiculo_involucrado -> hecho,PASS,0,0
5,fk_check,vehiculo_involucrado -> vehiculo,PASS,0,0
6,fk_check,fallecido_lesionado -> hecho,PASS,0,0
7,fk_check,fallecido_lesionado -> vehiculo,PASS,0,0
8,postgis_check,departamento_geom.geometry_valid,PASS,0,0
9,postgis_check,municipio_geom.geometry_valid,PASS,0,0


2026-07-13 23:12:03 | INFO | etl_load_database | Validaciones finales ejecutadas: 11
2026-07-13 23:12:03 | INFO | etl_load_database | Resultados FAIL: 0
2026-07-13 23:12:03 | INFO | etl_load_database | Resultados WARN: 0
2026-07-13 23:12:03 | INFO | etl_load_database | Seccion 12 finalizada correctamente.


## Seccion 13. Deteccion y registro de inconsistencias de calidad

Esta seccion detecta y registra inconsistencias conocidas en datos oficiales sin autocorreccion.

Casos controlados:
- `hora_ocu = 99`,
- `color_veh = 999`,
- `zona_ocu = 0`,
- registros con `floor(mupio_ocu/100) != depto_ocu` en `fallecidos_lesionados`.

Todas las incidencias se documentan en `df_quality_issues` para reporte final.

In [45]:
# ================================================================
# Registro de calidad de datos (sin autocorreccion)
# ================================================================

LOGGER.info("Iniciando deteccion de inconsistencias de calidad...")

quality_rows = []


def append_quality_rows(
    df: pd.DataFrame,
    source_file: str,
    condition: pd.Series,
    field_name: str,
    rule_violated: str,
    issue_type: str,
    severity: str = "WARNING",
    origin: str = "staging_detected",
) -> None:
    """Agrega filas de incidencias de calidad al acumulador quality_rows."""
    flagged = df.loc[condition].copy()
    if flagged.empty:
        return

    for row in flagged.itertuples(index=False):
        record_key = f"num_corre={int(row.num_corre)}|anio_ocu={int(row.ano_ocu)}"
        observed_value = getattr(row, field_name)
        quality_rows.append(
            {
                "source_file": source_file,
                "record_key": record_key,
                "field_name": field_name,
                "observed_value": observed_value,
                "rule_violated": rule_violated,
                "issue_type": issue_type,
                "origin": origin,
                "action_taken": "logged_no_autocorrect",
                "severity": severity,
            }
        )


# hora_ocu = 99 (codigo ignorado oficial observado)
append_quality_rows(
    df_hechos_stg,
    "hechos_clean.csv",
    df_hechos_stg["hora_ocu"] == 99,
    "hora_ocu",
    "hora_ocu fuera de rango horario 0..23",
    "domain_anomaly",
)
append_quality_rows(
    df_vi_stg,
    "vehiculos_involucrados_clean.csv",
    df_vi_stg["hora_ocu"] == 99,
    "hora_ocu",
    "hora_ocu fuera de rango horario 0..23",
    "domain_anomaly",
)
append_quality_rows(
    df_fl_stg,
    "fallecidos_lesionados_clean.csv",
    df_fl_stg["hora_ocu"] == 99,
    "hora_ocu",
    "hora_ocu fuera de rango horario 0..23",
    "domain_anomaly",
)

# color_veh = 999 (difiere de codigo ignorado esperado 99)
append_quality_rows(
    df_hechos_stg,
    "hechos_clean.csv",
    df_hechos_stg["color_veh"] == 999,
    "color_veh",
    "codigo de color no contemplado en catalogo oficial",
    "domain_anomaly",
)

# zona_ocu = 0 (codigo no esperado)
append_quality_rows(
    df_fl_stg,
    "fallecidos_lesionados_clean.csv",
    df_fl_stg["zona_ocu"] == 0,
    "zona_ocu",
    "codigo de zona fuera de dominio esperado",
    "domain_anomaly",
)

# Casos verificados en fuente INE: floor(mupio_ocu/100) != depto_ocu
verified_source_errors = {
    (2019, 10387),
    (2019, 10388),
    (2019, 10573),
    (2024, 764),
    (2024, 765),
}

mismatch_condition = (df_fl_stg["mupio_ocu"] // 100) != df_fl_stg["depto_ocu"]
df_mismatch = df_fl_stg.loc[mismatch_condition].copy()
for row in df_mismatch.itertuples(index=False):
    row_key = (int(row.ano_ocu), int(row.num_corre))
    quality_rows.append(
        {
            "source_file": "fallecidos_lesionados_clean.csv",
            "record_key": f"num_corre={int(row.num_corre)}|anio_ocu={int(row.ano_ocu)}",
            "field_name": "mupio_ocu,depto_ocu",
            "observed_value": f"mupio_ocu={int(row.mupio_ocu)},depto_ocu={int(row.depto_ocu)}",
            "rule_violated": "floor(mupio_ocu/100) debe ser igual a depto_ocu",
            "issue_type": "geo_consistency_anomaly",
            "origin": "source_verified" if row_key in verified_source_errors else "staging_detected",
            "action_taken": "logged_no_autocorrect",
            "severity": "WARNING",
        }
    )

df_quality_issues = pd.DataFrame(quality_rows)
if df_quality_issues.empty:
    df_quality_issues = pd.DataFrame(
        columns=[
            "source_file",
            "record_key",
            "field_name",
            "observed_value",
            "rule_violated",
            "issue_type",
            "origin",
            "action_taken",
            "severity",
        ]
    )

LOGGER.warning("Inconsistencias de calidad registradas: %s", len(df_quality_issues))
display(df_quality_issues.head(50))
LOGGER.info("Seccion 13 finalizada correctamente.")


2026-07-13 21:55:22 | INFO | etl_load_database | Iniciando deteccion de inconsistencias de calidad...
2026-07-13 21:55:22 | WARNING | etl_load_database | Inconsistencias de calidad registradas: 118


,source_file,record_key,field_name,observed_value,rule_violated,issue_type,origin,action_taken,severity
0,hechos_clean.csv,num_corre=3991|anio_ocu=2021,hora_ocu,99,hora_ocu fuera de rango horario 0..23,domain_anomaly,staging_detected,logged_no_autocorrect,WARNING
1,hechos_clean.csv,num_corre=4002|anio_ocu=2021,hora_ocu,99,hora_ocu fuera de rango horario 0..23,domain_anomaly,staging_detected,logged_no_autocorrect,WARNING
2,hechos_clean.csv,num_corre=4050|anio_ocu=2021,hora_ocu,99,hora_ocu fuera de rango horario 0..23,domain_anomaly,staging_detected,logged_no_autocorrect,WARNING
3,hechos_clean.csv,num_corre=6250|anio_ocu=2021,hora_ocu,99,hora_ocu fuera de rango horario 0..23,domain_anomaly,staging_detected,logged_no_autocorrect,WARNING
4,hechos_clean.csv,num_corre=3973|anio_ocu=2022,hora_ocu,99,hora_ocu fuera de rango horario 0..23,domain_anomaly,staging_detected,logged_no_autocorrect,WARNING
5,hechos_clean.csv,num_corre=3974|anio_ocu=2022,hora_ocu,99,hora_ocu fuera de rango horario 0..23,domain_anomaly,staging_detected,logged_no_autocorrect,WARNING
6,hechos_clean.csv,num_corre=6075|anio_ocu=2022,hora_ocu,99,hora_ocu fuera de rango horario 0..23,domain_anomaly,staging_detected,logged_no_autocorrect,WARNING
7,hechos_clean.csv,num_corre=6097|anio_ocu=2022,hora_ocu,99,hora_ocu fuera de rango horario 0..23,domain_anomaly,staging_detected,logged_no_autocorrect,WARNING
8,hechos_clean.csv,num_corre=6808|anio_ocu=2022,hora_ocu,99,hora_ocu fuera de rango horario 0..23,domain_anomaly,staging_detected,logged_no_autocorrect,WARNING
9,hechos_clean.csv,num_corre=6810|anio_ocu=2022,hora_ocu,99,hora_ocu fuera de rango horario 0..23,domain_anomaly,staging_detected,logged_no_autocorrect,WARNING


2026-07-13 21:55:22 | INFO | etl_load_database | Seccion 13 finalizada correctamente.


In [54]:
gdf = gpd.read_file("../data/IDEG/agrip_04_Limites_municipales_340.json")

print(gdf.crs)
print(gdf.total_bounds)

EPSG:4326
[-92.24019453  13.73859608 -88.22157516  17.81758601]
